# Interval Selection for Adversarial Rank Suppression

This notebook accompanies the paper's two evaluation rounds: an initial **48 queries** and a prospectively locked **93-query extension** on the same 873-video collection. The pooled 141-query analysis is secondary.

Keep `MODE = "reported_summary"` and run all cells for the lightweight reviewer path. It needs no GPU, Drive, dataset or repository access. It presents the supplied aggregate results and recomputes the exact sign-test arithmetic; it does not rerun attacks or independently verify the original authorities.

The notebook starts from promoted and locked artifacts. Human review, raw videos, keyframes, weights, caches and execution authorities are external and are not bundled. The exact historical runtime is embedded, so no GitHub token is needed. Full audit or replay requires authorized access to the original Drive assets, and GPU attack stages require at least 18 GiB of free CUDA memory.

Round 2 began with 100 video-grounded queries. Four ambiguous cases and three development-video overlaps were excluded, leaving 93. Round 1 used inherited/translated English prompts; round 2 used authored English prompts. The second configuration was locked after round-1 results were known but before round-2 attacks.


In [ ]:
MODE = "reported_summary"
SELECTED_STAGES = ["AG42"]

import os
from pathlib import Path
import base64
import hashlib
import json
import subprocess
import sys

WORK_DIR = Path(os.environ.get("ISARS_WORK_DIR", "isars_reproduction_work")).resolve()
ARTIFACT_ROOT = Path(os.environ.get("ISARS_ARTIFACT_ROOT", "artifacts")).expanduser().resolve()
DATA_ROOT = Path(os.environ.get("ISARS_DATA_ROOT", "dataset")).expanduser().resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)


## 1. Method implementation

The embedded runtime contains the normalized prompt/frame pooling, max-window ranking, three contiguous interval selectors, and the 15-step PGD implementation described in the paper. It is preserved byte-for-byte from commit `ab526c50f491bae440ea697fca72adcb10c687f3` and contains code only.

| Component | Frozen stages |
|---|---|
| Split, prompt and configuration locks | AG30, AG32, AG34, AG37R, AG38, AG40 |
| Contiguous interval selection | AG33, AG39 |
| GPU evaluation | AG35, AG41 |
| Result audit and statistics | AG36, AG42 |

The stage cells below register readable source without executing it. Comments that affect source hashes or preserved function bodies remain unchanged.


In [ ]:
SOURCE_COMMIT = 'ab526c50f491bae440ea697fca72adcb10c687f3'
RUNTIME_SHA256 = '2b51b72c54d202516ccd2cd60c2f28e955db57c3e0ed8859e3f10589618339d6'
RUNTIME_BASE64 = '...'
STAGE_MANIFEST = {'AG30': {'original_file': 'docs/AG30_ORGANIZER58_DEV10_TEST48_SPLIT_LOCK_COLAB.py', 'original_sha256': '8e6b74fd55916554d845eb9a7eaee547c953dee7b72b0c3a54376ad439fca314', 'notebook_source_sha256': '4fc87dd68d140b2cc267de654e668a1687be68672614b1c9a9f60cd29d42dedd', 'adaptation': 'comments removed outside functions; storage roots and timeline folders replaced by semantic aliases'}, 'AG32': {'original_file': 'docs/AG32_TEST48_PROMPT_CENTROID_EXPANSION_COLAB.py', 'original_sha256': '53939a81dc6f1c4f87b864fff0f0657b8da6a5f3a73a26ca432d70868c58f53d', 'notebook_source_sha256': '758117498a9e7ff74bce65946843acb8243b82c5a8ba7aba3053b78c62443323', 'adaptation': 'comments removed outside functions; storage roots and timeline folders replaced by semantic aliases'}, 'AG33': {'original_file': 'docs/AG33_TEST48_REGION_SELECTION_COLAB.py', 'original_sha256': '635d9a3965645fd7bebb48c152cbcf9c430ccdc887537bc872ce5aa768590858', 'notebook_source_sha256': 'af9f026e2433051f8db55f1219a9b93a2bcc7a35645a9e772a48601982d3341f', 'adaptation': 'comments removed outside functions; storage roots and timeline folders replaced by semantic aliases'}, 'AG34': {'original_file': 'docs/AG34_TEST48_FINAL_PRE_EVALUATION_LOCK_COLAB.py', 'original_sha256': '4434e3f62007544fdaaa24c52182b7b2b2b97a4e0455418d11585f25662169f1', 'notebook_source_sha256': '9974cde982cbafb86f4e8b7cfa7ccdaf3fbdc91d04df78d37f8e1e4e7f4154c7', 'adaptation': 'comments removed outside functions; storage roots and timeline folders replaced by semantic aliases'}, 'AG35': {'original_file': 'docs/AG35_FINAL_TEST48_REGION_EVALUATION_COLAB.py', 'original_sha256': 'e9efa53717caba2bcba18a2e2b6ff5d116ffac9494c5808bee7bec34b0e7b684', 'notebook_source_sha256': '699d0c0c9118aefd51822499c6d891ee941a1899bd692c81aaa6ecf13808010a', 'adaptation': 'comments removed outside functions; storage roots and timeline folders replaced by semantic aliases'}, 'AG36': {'original_file': 'docs/AG36_TEST48_RESULT_AUDIT_COLAB.py', 'original_sha256': '8646a3bac9310eecc7d893dbece45b00d9f1751c7319153314724d52e77bd506', 'notebook_source_sha256': '8861b6aebedfd19e10bccd139659df6a21e555a3fc822787e7bc63de8908cacb', 'adaptation': 'comments removed outside functions; storage roots and timeline folders replaced by semantic aliases'}, 'AG37R': {'original_file': 'docs/AG37R_BATCH_A_TEST48_EXTENSION_LOCK_V2_COLAB.py', 'original_sha256': '16ef692bc5bd05ad23dc4f3fd8320f326150e33fa00420a95e63b4c294fc9974', 'notebook_source_sha256': '1884d0a3252ca1cf6cf11a7bd99f4c95a55cd00d45240e5da2b4a87b56e8988c', 'adaptation': 'comments removed outside functions; storage roots and timeline folders replaced by semantic aliases'}, 'AG38': {'original_file': 'docs/AG38_BATCH_A_EXTENSION_PROMPT_CATALOG_LOCK_COLAB.py', 'original_sha256': 'b3bc519aeefc4521727146b1aceaeac5d0faa3c5ff06042e6cb4741b87dcf7aa', 'notebook_source_sha256': '9a591b201d64ad2feb1d7568c1a26a380b5e0b0807f5b5cdd26ceeee51fd58c3', 'adaptation': 'comments removed outside functions; storage roots and timeline folders replaced by semantic aliases'}, 'AG39': {'original_file': 'docs/AG39_BATCH_A_EXTENSION_REGION_SELECTION_COLAB.py', 'original_sha256': '804e1831e56aad01e01d6818392cf07ef9eefd73cd6dde46fdd02c52e5d8845a', 'notebook_source_sha256': '1d96f392aafb7020b77aae3b6fb05a4a29cc8dfd9db5e9f82331c57857e60e93', 'adaptation': 'comments removed outside functions; storage roots and timeline folders replaced by semantic aliases'}, 'AG40': {'original_file': 'docs/AG40_BATCH_A_FINAL_PRE_EVALUATION_LOCK_COLAB.py', 'original_sha256': '4ca09712fabe5b23834c6c83a69263e3978499de0745cef43c6b5a682e7fb999', 'notebook_source_sha256': 'e9de0b18962b82f6c309ea754047ae7ad760efcfc8bd8bbcf9aaebad0988daff', 'adaptation': 'comments removed outside functions; storage roots and timeline folders replaced by semantic aliases'}, 'AG41': {'original_file': 'docs/AG41_BATCH_A_EXTENSION_GPU_EVALUATION_COLAB.py', 'original_sha256': '9aac6c34ede1c4aafdc136ee803f5087407f71a5bdc4cce98f4b62b24cf9a31c', 'notebook_source_sha256': 'f6f4d7babf1cff951c518d6374d18570d5bc7349f866431f11367a11d0e68552', 'adaptation': 'comments removed outside functions; storage roots and timeline folders replaced by semantic aliases'}, 'AG42': {'original_file': 'docs/AG42_BATCH_A_AND_POOLED_RESULT_AUDIT_COLAB.py', 'original_sha256': 'cc7b98cceae7fb6b306ae9d89bf71afc9d5d45d9fc4dcfbc83a529e9c91df0b2', 'notebook_source_sha256': 'c7b8d77e08c9eb401745a93622fd06185a9fe6c3aaa23e565123cc250763b0b9', 'adaptation': 'comments removed outside functions; storage roots and timeline folders replaced by semantic aliases'}}
runtime_path = WORK_DIR / "pinned_runtime.zip"
runtime_bytes = base64.b64decode(RUNTIME_BASE64)
assert hashlib.sha256(runtime_bytes).hexdigest() == RUNTIME_SHA256
runtime_path.write_bytes(runtime_bytes)


In [ ]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def stage(name, cell):
    name = name.strip()
    payload = cell.encode("utf-8")
    expected = STAGE_MANIFEST[name]["notebook_source_sha256"]
    if hashlib.sha256(payload).hexdigest() != expected:
        raise ValueError(f"Stage source changed: {name}")
    compile(cell, f"{name}.py", "exec")
    (WORK_DIR / f"{name}.py").write_bytes(payload)
    print(f"Registered {name}; execution deferred")


## 2. Reported evaluation

`reported_summary` prints the supplied audit summaries and independently recomputes exact sign-test p-values from their win/loss counts. It does not rederive ranks or confidence intervals from execution records.


In [ ]:
REPORTED_RESULTS = {'round1': {'artifact_class': 'reported_summary_of_test48_post_evaluation_audit', 'source_log_marker': 'AG36_TEST48_RESULT_AUDIT_OK', 'evidence_status': 'USER_REPORTED_COLAB_OUTPUT_NOT_INDEPENDENTLY_REHASHED_LOCALLY', 'audit_status': 'test48_post_evaluation_audit_passed', 'audit_path': 'ag36_test48_cpu_result_audit_v1.json', 'audit_sha256': 'a82b50ff4b32e3f8bd37dd03ca3210490266060bc635b441029c5a697786ca81', 'ag35_comparison_sha256': 'e688273cb949b53728d5434d4426f50320935004baa22fa82e58db9323689a15', 'ag35_protocol_sha256': '5fc4c619070ed6fb6dd79ccd0d445ffccaad25a74d090d6eff510a52452d9b0c', 'test48_sha256': '231ff1f0432be2bbc1eee9f879a5284f3d6c7633c5c674fdabd605495fd607f5', 'query_count': 48, 'video_count': 46, 'successful_execution_count': 144, 'failure_count': 0, 'paired_denominator': 48, 'metrics': {'target_centered': {'mean_rank': 45.0625, 'median_rank': 21.0, 'mrr': 0.2137499423331832, 'recall_at_10': 0.3541666666666667, 'asr_at_10': '12/29', 'mean_psnr': 55.59818917827621, 'mean_ssim': 0.9986389841977946}, 'highest_score_window': {'mean_rank': 70.16666666666667, 'median_rank': 27.0, 'mrr': 0.1749038977480006, 'recall_at_10': 0.3541666666666667, 'asr_at_10': '12/29', 'mean_psnr': 55.57941005948842, 'mean_ssim': 0.9986638757827562}, 'minimum_unaffected_floor': {'mean_rank': 86.64583333333333, 'median_rank': 28.0, 'mrr': 0.15950157312810334, 'recall_at_10': 0.3541666666666667, 'asr_at_10': '12/29', 'mean_psnr': 56.41421839981035, 'mean_ssim': 0.9988157185590786}}, 'paired_comparisons': {'versus_target_centered': {'wins_ties_losses': [24, 23, 1], 'non_tied_denominator': 25, 'mean_rank_delta': 41.583333333333336, 'rank_delta_ci95_video_cluster': [18.419193877551024, 71.54372109158186], 'mean_mrr_delta': -0.05424836920507986, 'mrr_delta_ci95_video_cluster': [-0.10819649491264825, -0.013028974525772338], 'exact_sign_test_p_unadjusted': 1.5497207641601562e-06, 'bonferroni_adjusted_p': 3.0994415283203125e-06, 'significant_familywise_0_05': True}, 'versus_highest_score_window': {'wins_ties_losses': [15, 32, 1], 'non_tied_denominator': 16, 'mean_rank_delta': 16.479166666666668, 'rank_delta_ci95_video_cluster': [4.4891068262411356, 33.69611801242235], 'mean_mrr_delta': -0.015402324619897262, 'mrr_delta_ci95_video_cluster': [-0.03890565909560401, -0.002079286716379094], 'exact_sign_test_p_unadjusted': 0.000518798828125, 'bonferroni_adjusted_p': 0.00103759765625, 'significant_familywise_0_05': True}}, 'claim_boundary': 'Under the frozen max-window dense-retrieval protocol, minimum-unaffected-floor increased target-rank suppression severity relative to both matched-budget selectors, while all three strategies retained the same Recall@10 and ASR@10 of 12/29.'}, 'round2': {'provenance': 'USER_REPORTED_COLAB_OUTPUT_RECOMPUTED_FROM_HASH_BOUND_AUTHORITIES_BY_AG42', 'recorded_date': '2026-09-11', 'marker': 'AG42_BATCH_A_AND_POOLED_RESULT_AUDIT_OK', 'audit_sha256': '5b9155a071b3949bdf01352aea38b103d9917e2a6887bf525f6d40c90a37979e', 'audit_path': 'ag42_batch_a_and_pooled_cpu_result_audit_v1.json', 'successful_execution_count': 423, 'failure_count': 0, 'historical_test48_reproduced': True, 'batch_a_extension': {'query_count': 93, 'video_count': 93, 'paired_denominator': 93, 'bootstrap': {'method': 'paired video-cluster bootstrap', 'cluster_count': 93, 'query_weighted': True, 'draws': 10000, 'seed': 20260911, 'interval': 'percentile 95%'}, 'metrics': {'target_centered': {'mean_rank': 53.26881720430107, 'median_rank': 11.0, 'mrr': 0.2880061646112341, 'recall_at_1': 0.1935483870967742, 'recall_at_5': 0.3548387096774194, 'recall_at_10': 0.4838709677419355, 'mean_rank_shift_from_clean': 32.27956989247312, 'median_rank_shift_from_clean': 0.0, 'asr_at_10_numerator': 13, 'asr_at_10_denominator': 58, 'mean_psnr': 55.90931611371345, 'mean_ssim': 0.9985799885638086, 'mean_elapsed_seconds': 152.48394240487085}, 'highest_score_window': {'mean_rank': 64.25806451612904, 'median_rank': 21.0, 'mrr': 0.20123677106237517, 'recall_at_1': 0.10752688172043011, 'recall_at_5': 0.26881720430107525, 'recall_at_10': 0.3118279569892473, 'mean_rank_shift_from_clean': 43.26881720430107, 'median_rank_shift_from_clean': 10.0, 'asr_at_10_numerator': 29, 'asr_at_10_denominator': 58, 'mean_psnr': 55.853806630857406, 'mean_ssim': 0.9986979769114522, 'mean_elapsed_seconds': 105.01819160616114}, 'minimum_unaffected_floor': {'mean_rank': 65.91397849462365, 'median_rank': 22.0, 'mrr': 0.17802602199055537, 'recall_at_1': 0.0967741935483871, 'recall_at_5': 0.23655913978494625, 'recall_at_10': 0.27956989247311825, 'mean_rank_shift_from_clean': 44.924731182795696, 'median_rank_shift_from_clean': 11.0, 'asr_at_10_numerator': 32, 'asr_at_10_denominator': 58, 'mean_psnr': 56.70237782680651, 'mean_ssim': 0.9989491176546479, 'mean_elapsed_seconds': 105.25549838009692}}, 'paired_comparisons': {'target_centered': {'direction': 'positive rank delta means stronger target-rank suppression by minimum_unaffected_floor', 'wins_ties_losses': [61, 32, 0], 'non_tied_denominator': 61, 'mean_rank_delta': 12.64516129032258, 'median_rank_delta': 3.0, 'rank_delta_ci95_video_cluster': [7.870698924731183, 18.666935483870965], 'mean_mrr_delta': -0.10998014262067875, 'mrr_delta_ci95_video_cluster': [-0.15366125177889928, -0.06946788167040405], 'exact_two_sided_sign_test_p_unadjusted': 8.673617379884035e-19, 'bonferroni_family_size': 2, 'bonferroni_adjusted_p': 1.734723475976807e-18, 'familywise_alpha': 0.05, 'per_comparison_alpha': 0.025, 'significant_familywise_0_05': True}, 'highest_score_window': {'direction': 'positive rank delta means stronger target-rank suppression by minimum_unaffected_floor', 'wins_ties_losses': [19, 74, 0], 'non_tied_denominator': 19, 'mean_rank_delta': 1.6559139784946237, 'median_rank_delta': 0.0, 'rank_delta_ci95_video_cluster': [0.7634408602150538, 2.78494623655914], 'mean_mrr_delta': -0.023210749071819824, 'mrr_delta_ci95_video_cluster': [-0.045126374387395564, -0.006586313289266481], 'exact_two_sided_sign_test_p_unadjusted': 3.814697265625e-06, 'bonferroni_family_size': 2, 'bonferroni_adjusted_p': 7.62939453125e-06, 'familywise_alpha': 0.05, 'per_comparison_alpha': 0.025, 'significant_familywise_0_05': True}}}, 'pooled_141_secondary': {'query_count': 141, 'video_count': 130, 'paired_denominator': 141, 'bootstrap': {'method': 'paired video-cluster bootstrap', 'cluster_count': 130, 'query_weighted': True, 'draws': 10000, 'seed': 20260911, 'interval': 'percentile 95%'}, 'metrics': {'target_centered': {'mean_rank': 50.47517730496454, 'median_rank': 14.0, 'mrr': 0.2627274506442381, 'recall_at_1': 0.1702127659574468, 'recall_at_5': 0.3475177304964539, 'recall_at_10': 0.4397163120567376, 'mean_rank_shift_from_clean': 29.29787234042553, 'median_rank_shift_from_clean': 0.0, 'asr_at_10_numerator': 25, 'asr_at_10_denominator': 87, 'mean_psnr': 55.803400561224166, 'mean_ssim': 0.9986000721838889, 'mean_elapsed_seconds': 152.83646470807088}, 'highest_score_window': {'mean_rank': 66.26950354609929, 'median_rank': 23.0, 'mrr': 0.19227238865748167, 'recall_at_1': 0.10638297872340426, 'recall_at_5': 0.2695035460992908, 'recall_at_10': 0.3262411347517731, 'mean_rank_shift_from_clean': 45.09219858156028, 'median_rank_shift_from_clean': 10.0, 'asr_at_10_numerator': 41, 'asr_at_10_denominator': 87, 'mean_psnr': 55.76039503209349, 'mean_ssim': 0.9986863680165772, 'mean_elapsed_seconds': 103.4644343086453}, 'minimum_unaffected_floor': {'mean_rank': 72.97163120567376, 'median_rank': 23.0, 'mrr': 0.17171982663312488, 'recall_at_1': 0.09219858156028368, 'recall_at_5': 0.24822695035460993, 'recall_at_10': 0.3049645390070922, 'mean_rank_shift_from_clean': 51.794326241134755, 'median_rank_shift_from_clean': 11.0, 'asr_at_10_numerator': 44, 'asr_at_10_denominator': 87, 'mean_psnr': 56.60428100059505, 'mean_ssim': 0.9989037051965817, 'mean_elapsed_seconds': 104.39291772426247}}, 'paired_comparisons': {'target_centered': {'direction': 'positive rank delta means stronger target-rank suppression by minimum_unaffected_floor', 'wins_ties_losses': [85, 55, 1], 'non_tied_denominator': 86, 'mean_rank_delta': 22.49645390070922, 'median_rank_delta': 2.0, 'rank_delta_ci95_video_cluster': [13.474726443768997, 33.08045858457632], 'mean_mrr_delta': -0.09100762401111315, 'mrr_delta_ci95_video_cluster': [-0.12712715296654875, -0.05797079474079972], 'exact_two_sided_sign_test_p_unadjusted': 2.248897290378544e-24, 'bonferroni_family_size': 2, 'bonferroni_adjusted_p': 4.497794580757088e-24, 'familywise_alpha': 0.05, 'per_comparison_alpha': 0.025, 'significant_familywise_0_05': True}, 'highest_score_window': {'direction': 'positive rank delta means stronger target-rank suppression by minimum_unaffected_floor', 'wins_ties_losses': [34, 106, 1], 'non_tied_denominator': 35, 'mean_rank_delta': 6.702127659574468, 'median_rank_delta': 0.0, 'rank_delta_ci95_video_cluster': [2.4717484979808924, 12.695194296690305], 'mean_mrr_delta': -0.020552562024356825, 'mrr_delta_ci95_video_cluster': [-0.03680459073636678, -0.00729454505711382], 'exact_two_sided_sign_test_p_unadjusted': 2.0954757928848267e-09, 'bonferroni_family_size': 2, 'bonferroni_adjusted_p': 4.190951585769653e-09, 'familywise_alpha': 0.05, 'per_comparison_alpha': 0.025, 'significant_familywise_0_05': True}}}}}
HARNESS = 'import io\nimport runpy\nimport sys\nimport urllib.request\nfrom pathlib import Path\nfrom google.colab import drive, userdata\n\nruntime_archive = Path(sys.argv[2]).read_bytes()\noriginal_urlopen = urllib.request.urlopen\noriginal_get = userdata.get\n\ndef source_urlopen(request, *args, **kwargs):\n    url = request.full_url if hasattr(request, "full_url") else str(request)\n    if url == "https://api.github.com/repos/haruxne/STV-Attack/zipball/ab526c50f491bae440ea697fca72adcb10c687f3":\n        return io.BytesIO(runtime_archive)\n    return original_urlopen(request, *args, **kwargs)\n\ndef source_credential(name, *args, **kwargs):\n    if name == "GITHUB_TOKEN":\n        return "embedded-runtime-no-network-credential"\n    return original_get(name, *args, **kwargs)\n\nurllib.request.urlopen = source_urlopen\nuserdata.get = source_credential\ndrive.mount = lambda *args, **kwargs: None\nrunpy.run_path(sys.argv[1], run_name="__main__")\n'


In [ ]:
import math

def print_reported():
    report = REPORTED_RESULTS["round2"]
    print("REPORTED_SUMMARY_ONLY: original Drive artifacts not checked in this mode")
    for label, cohort in [("Round 1: 48", REPORTED_RESULTS["round1"]),
                          ("Round 2: 93", report["batch_a_extension"]),
                          ("Pooled: 141 (secondary)", report["pooled_141_secondary"])]:
        print("\n" + label)
        for strategy, values in cohort["metrics"].items():
            print(strategy, "mean rank", round(values["mean_rank"], 2),
                  "MRR", round(values["mrr"], 3),
                  "ASR@10", values["asr_at_10"] if "asr_at_10" in values else
                  f"{values['asr_at_10_numerator']}/{values['asr_at_10_denominator']}")
    for baseline, comparison in report["batch_a_extension"]["paired_comparisons"].items():
        wins, ties, losses = comparison["wins_ties_losses"]
        n = wins + losses
        p = min(1.0, 2 * sum(math.comb(n, k) for k in range(min(wins, losses) + 1)) / 2**n)
        assert p == comparison["exact_two_sided_sign_test_p_unadjusted"]
        assert min(1.0, 2 * p) == comparison["bonferroni_adjusted_p"]
        print(baseline, "W/T/L", (wins, ties, losses), "adjusted p", min(1.0, 2 * p))

if MODE == "reported_summary":
    print_reported()
elif MODE not in {"audit", "selected_stages"}:
    raise ValueError("MODE must be reported_summary, audit, or selected_stages")


## 3. Optional audit and GPU replay

Use `MODE = "audit"` to run AG42 against the original Drive authorities and pinned hashes. Use `selected_stages` only when the required locked inputs already exist; for example, `SELECTED_STAGES = ["AG39", "AG40", "AG41", "AG42"]`. Each selected stage runs in a separate process.

Set `ISARS_ARTIFACT_ROOT` and `ISARS_DATA_ROOT` before running the configuration cell, or edit `ARTIFACT_ROOT` and `DATA_ROOT` there. The notebook maps those locations to neutral runtime aliases; it does not assume the authors' storage layout. Audit needs both run authorities, comparison files, the saved round-1 audit and all inputs checked by the final audit stage. Missing evidence stops execution. Original hashes must not be replaced to force a pass.

AG35 and AG41 are expensive CUDA replays. They are included for implementation transparency and are not part of the default reviewer path. Fresh runs may differ in timing and output hashes from the recorded execution.


In [ ]:
%%stage AG30
                                                                                            
                                                           
                                                                                      

import hashlib
import json
import os
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

ROOT = Path("/content/reproduction_inputs/artifacts")
OLD_ROOT = ROOT / "evaluation/round1_method_lock"
SOURCE_ROOT = ROOT / "evaluation/extension_query_promotion"
OUTPUT = ROOT / "evaluation/round1_evaluation"

OLD_DEV = OLD_ROOT / "splits_seed_20260904/dev.json"
AG11_LOCK = OLD_ROOT / "dev10_centroid_attack_suite/pre_evaluation_configuration_lock.json"
MERGED = SOURCE_ROOT / "organizer58_promotion_ready_annotations.jsonl"
AG29_MANIFEST = SOURCE_ROOT / "organizer58_promotion_ready_manifest.json"

DEV = OUTPUT / "dev10.json"
TEST = OUTPUT / "test48.json"
MANIFEST = OUTPUT / "dev10_test48_split_lock_manifest.json"

EXPECTED_OLD_DEV_SHA256 = "826ba5df23abcfd061deb8c4763c56ebdf11fbae2f8902c7bf22cd66590ec067"
EXPECTED_MERGED_SHA256 = "71e43cadb55c884852b31c6297065ba8e59fdeb3ed92b532bcb68c17d2e7a246"
EXPECTED_AG29_MANIFEST_SHA256 = "a0ed6bd5f83138b9290d56a0892fe68a25b3057655d8e269dfdb214e7bc303ed"


def require(ok, message):
    if not ok:
        raise RuntimeError(message)


def sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def loadj(path):
    return json.loads(Path(path).read_bytes())


def loadjl(path):
    return [json.loads(line) for line in Path(path).read_text(
        encoding="utf-8-sig").splitlines() if line.strip()]


def pretty(value):
    return (json.dumps(value, ensure_ascii=False, indent=2,
                       sort_keys=True) + "\n").encode("utf-8")


def canon(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True,
                       separators=(",", ":")) + "\n").encode("utf-8")


def immutable_write(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        require(path.read_bytes() == payload, f"Immutable artifact drift: {path}")
        return
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, path)


for path in (OLD_DEV, AG11_LOCK, MERGED, AG29_MANIFEST):
    require(path.is_file(), f"Missing required artifact: {path}")
require(sha(OLD_DEV) == EXPECTED_OLD_DEV_SHA256, "Original dev-10 hash drift")
require(sha(MERGED) == EXPECTED_MERGED_SHA256, "AG29 merged cohort hash drift")
require(sha(AG29_MANIFEST) == EXPECTED_AG29_MANIFEST_SHA256, "AG29 manifest hash drift")

old_dev = loadj(OLD_DEV)
ag11 = loadj(AG11_LOCK)
ag29 = loadj(AG29_MANIFEST)
merged = loadjl(MERGED)

require(isinstance(old_dev, list) and len(old_dev) == 10, "Original dev split is not dev-10")
require(ag11.get("status") == "development_configuration_selected_before_locked_evaluation",
        "AG11 configuration lock is invalid")
require(ag11.get("selected_condition") == "first_order", "Unexpected selected method")
require(ag11.get("test_split_touched") is False, "AG11 touched held-out data")
require(ag29.get("status") == "organizer58_promotion_ready_pending_split_lock",
        "AG29 cohort is not promotion-ready")
require(ag29.get("merged_promoted", {}).get("sha256") == sha(MERGED),
        "AG29 manifest does not bind merged bytes")
require(ag29.get("ground_truth_promoted") is True, "AG29 cohort is not promoted")
require(ag29.get("test_split_touched") is False, "AG29 unexpectedly touched a test split")
require(len(merged) == 58, "Expected exactly 58 promoted records")

merged_by_id = {row["query_id"]: row for row in merged}
require(len(merged_by_id) == 58, "Duplicate query IDs in merged cohort")
dev_ids = {row["query_id"] for row in old_dev}
require(len(dev_ids) == 10, "Duplicate query IDs in original dev-10")
require(dev_ids.issubset(merged_by_id), "Original dev-10 is not a subset of organizer58")

                                                                          
dev = sorted((merged_by_id[query_id] for query_id in dev_ids),
             key=lambda row: row["query_id"])
test = sorted((row for row in merged if row["query_id"] not in dev_ids),
              key=lambda row: row["query_id"])
require(len(dev) == 10 and len(test) == 48, "Split must be exactly dev10/test48")

                                                                      
old_dev_by_id = {row["query_id"]: row for row in old_dev}
for row in dev:
    old = old_dev_by_id[row["query_id"]]
    for field in ("video_id", "target_keyframe_n", "target_frame_id"):
        require(row.get(field) == old.get(field),
                f"{row['query_id']}: promoted dev target changed in {field}")
    require(row.get("ground_truth_promoted") is True,
            f"{row['query_id']}: dev record is not promoted")
for row in test:
    require(row.get("ground_truth_promoted") is True,
            f"{row['query_id']}: test record is not promoted")

dev_videos = {row["video_id"] for row in dev}
test_videos = {row["video_id"] for row in test}
leaked_videos = sorted(dev_videos & test_videos)
if leaked_videos:
    blocked = {
        "status": "ag30_blocked_video_leakage_no_files_written",
        "leaked_videos": leaked_videos,
        "affected_dev_query_ids": sorted(
            row["query_id"] for row in dev if row["video_id"] in leaked_videos),
        "affected_test_query_ids": sorted(
            row["query_id"] for row in test if row["video_id"] in leaked_videos),
        "dev_membership_changed": False,
        "test_split_written": False,
    }
    print("AG30_DEV10_TEST48_SPLIT_BLOCKED")
    print(json.dumps(blocked, ensure_ascii=False, indent=2))
    raise RuntimeError("Video leakage blocks exact test-48; no split artifact was written")

require(not ({row["query_id"] for row in dev} & {row["query_id"] for row in test}),
        "Query leakage across dev/test")
require({row["query_id"] for row in dev + test} == set(merged_by_id),
        "Split loses or adds queries")

immutable_write(DEV, pretty(dev))
immutable_write(TEST, pretty(test))
manifest = {
    "schema_version": 1,
    "status": "organizer58_dev10_test48_locked_before_evaluation",
    "artifact_class": "promoted_video_disjoint_split_and_configuration_lock",
    "source": {
        "ag29_manifest_sha256": sha(AG29_MANIFEST),
        "organizer58_sha256": sha(MERGED),
        "original_dev10_sha256": sha(OLD_DEV),
        "ag11_configuration_lock_sha256": sha(AG11_LOCK),
    },
    "selected_condition": ag11["selected_condition"],
    "selected_condition_config_hash": ag11["selected_condition_config_hash"],
    "selected_attack_parameters": ag11["selected_attack_parameters"],
    "retrieval_parameters": ag11["retrieval_parameters"],
    "dev": {
        "path": str(DEV), "sha256": sha(DEV),
        "query_count": 10, "video_count": len(dev_videos),
        "membership_source": "unchanged_query_ids_from_original_dev10",
    },
    "test": {
        "path": str(TEST), "sha256": sha(TEST),
        "query_count": 48, "video_count": len(test_videos),
        "membership_source": "all_remaining_promoted_organizer58_queries",
    },
    "unassigned_query_count": 0,
    "video_disjoint": True,
    "query_disjoint": True,
    "uses_mock": False,
    "ground_truth_promoted": True,
    "locked_evaluation_ready": True,
    "evaluation_executed": False,
    "test_results_observed": False,
}
immutable_write(MANIFEST, canon(manifest))

print("AG30_ORGANIZER58_DEV10_TEST48_SPLIT_LOCK_OK")
print(json.dumps({**manifest, "manifest_path": str(MANIFEST),
                  "manifest_sha256": sha(MANIFEST)}, ensure_ascii=False, indent=2))


In [ ]:
%%stage AG32
                                                                                                       
                                                                                           

import hashlib
import importlib.metadata
import importlib.util
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive

drive.mount("/content/drive")

ROOT = Path("/content/reproduction_inputs/artifacts")
LOCK_ROOT = ROOT / "evaluation/round1_evaluation"
TEST = LOCK_ROOT / "test48.json"
SPLIT_LOCK = LOCK_ROOT / "dev10_test48_split_lock_manifest.json"
OLD_PROMPTS = ROOT / "development/round1_prompt_catalog/retrieval_prompt_catalog.jsonl"
TRANSLATOR_ROOT = ROOT / "artifacts/models/opus-mt-vi-en_c8d2853"
SIGLIP_TOKENIZER = ROOT / "artifacts/models/ViT-B-16-SigLIP_webli/tokenizer_cache"
OUTPUT = LOCK_ROOT / "prompt_centroid_v1"
NEW_PROMPTS = OUTPUT / "missing16_prompt_catalog.jsonl"
FULL_PROMPTS = OUTPUT / "test48_prompt_catalog.jsonl"
MANIFEST = OUTPUT / "test48_prompt_catalog_manifest.json"

EXPECTED_TEST_SHA256 = "231ff1f0432be2bbc1eee9f879a5284f3d6c7633c5c674fdabd605495fd607f5"
EXPECTED_SPLIT_LOCK_SHA256 = "c4bac6f9eefdbe072b1ee9f6a44d7fd88df863139967c594fda1798a71c6fa8b"
EXPECTED_OLD_PROMPT_SHA256 = "2ec9807d2fe1a31e90e16dc16214726b00d282de1d6fee2ed780f79357824d63"
TRANSLATOR_REPOSITORY = "Helsinki-NLP/opus-mt-vi-en"
TRANSLATOR_REVISION = "c8d2853"
PROMPT_CONTEXT_LIMIT = 64
MISSING_IDS = {
    "HCMC-B1-ORG-SOTUYEN3-P2-Q001", "HCMC-B1-ORG-SOTUYEN3-P2-Q002",
    "HCMC-B1-ORG-SOTUYEN3-P2-Q004", "HCMC-B1-ORG-SOTUYEN3-P2-Q007",
    "HCMC-B1-ORG-SOTUYEN3-P2-Q009", "HCMC-B1-ORG-SOTUYEN3-P2-Q010",
    "HCMC-B1-ORG-SOTUYEN3-P2-Q011", "HCMC-B1-ORG-SOTUYEN3-P2-Q013",
    "HCMC-B1-ORG-SOTUYEN3-P2-Q017", "HCMC-B1-ORG-SOTUYEN3-P2-Q019",
    "HCMC-B1-ORG-SOTUYEN3-P2-Q020", "HCMC-B1-ORG-SOTUYEN3-P2-Q022",
    "HCMC-B1-ORG-SOTUYEN3-P2-Q023", "HCMC-B1-ORG-SOTUYEN3-P2-Q026",
    "HCMC-B1-ORG-SOTUYEN3-P2-Q028", "HCMC-B1-ORG-SOTUYEN3-P2-Q036",
}


def require(ok, message):
    if not ok:
        raise RuntimeError(message)


def sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canon(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True,
                       separators=(",", ":")) + "\n").encode("utf-8")


def load_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(
        encoding="utf-8-sig").splitlines() if line.strip()]


def immutable_write(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        require(path.read_bytes() == payload, f"Immutable artifact drift: {path}")
        return
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, path)


def asset_manifest(root):
    root = Path(root)
    excluded = {".cache", "__pycache__"}
    files = [path for path in root.rglob("*")
             if path.is_file() and not excluded.intersection(path.relative_to(root).parts)]
    require(files, f"No model assets found under {root}")
    entries = [{"path": path.relative_to(root).as_posix(), "size_bytes": path.stat().st_size,
                "sha256": sha(path)} for path in sorted(files)]
    return entries, hashlib.sha256(canon(entries)).hexdigest()


def provision_translator():
    required = [TRANSLATOR_ROOT / name for name in
                ("config.json", "pytorch_model.bin", "source.spm", "target.spm", "vocab.json")]
    proof = TRANSLATOR_ROOT / "ag03r_resolved_revision.json"
    if not all(path.is_file() for path in required):
        TRANSLATOR_ROOT.mkdir(parents=True, exist_ok=True)
        code = r'''
import json, sys
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
repo, revision, destination, proof = sys.argv[1:]
info = HfApi().model_info(repo, revision=revision)
assert info.sha.startswith(revision)
snapshot_download(repo_id=repo, revision=info.sha, local_dir=destination,
                  allow_patterns=["config.json", "generation_config.json", "pytorch_model.bin",
                                  "source.spm", "target.spm", "tokenizer_config.json", "vocab.json"])
Path(proof).write_text(json.dumps({"resolved_commit": info.sha}, sort_keys=True)+"\n")
'''
        environment = os.environ.copy()
        environment.pop("HF_HUB_OFFLINE", None)
        environment.pop("TRANSFORMERS_OFFLINE", None)
        subprocess.check_call([sys.executable, "-c", code, TRANSLATOR_REPOSITORY,
                               TRANSLATOR_REVISION, str(TRANSLATOR_ROOT), str(proof)], env=environment)
    require(proof.is_file(), "Missing pinned translator revision proof")
    resolved = json.loads(proof.read_text(encoding="utf-8"))["resolved_commit"]
    require(resolved.startswith(TRANSLATOR_REVISION), "Translator revision drift")
    entries, assets_sha = asset_manifest(TRANSLATOR_ROOT)
    return resolved, entries, assets_sha


def segments(text):
    result = [part.strip(" -") for part in
              re.split(r"(?<=[.!?])\s+|\s*[;•]\s+|\s+-\s+", text) if part.strip(" -")]
    require(result, "Cannot segment Vietnamese query")
    return result


for path in (TEST, SPLIT_LOCK, OLD_PROMPTS, SIGLIP_TOKENIZER):
    require(path.exists(), f"Missing required artifact: {path}")
require(sha(TEST) == EXPECTED_TEST_SHA256, "Locked test-48 hash drift")
require(sha(SPLIT_LOCK) == EXPECTED_SPLIT_LOCK_SHA256, "AG30 lock hash drift")
require(sha(OLD_PROMPTS) == EXPECTED_OLD_PROMPT_SHA256, "Old prompt catalog hash drift")

test = json.loads(TEST.read_bytes())
test_by_id = {row["query_id"]: row for row in test}
old = load_jsonl(OLD_PROMPTS)
old_by_id = {row["query_id"]: row for row in old}
require(len(test_by_id) == 48, "Expected 48 unique test queries")
observed_missing = set(test_by_id) - set(old_by_id)
require(observed_missing == MISSING_IDS,
        f"Missing prompt set changed: {sorted(observed_missing)}")

missing_packages = []
for module_name, package_name in (
    ("transformers", "transformers"),
    ("sentencepiece", "sentencepiece"),
    ("sacremoses", "sacremoses"),
    ("open_clip", "open_clip_torch==3.3.0"),
):
    if importlib.util.find_spec(module_name) is None:
        missing_packages.append(package_name)
if missing_packages:
    print("AG32_INSTALLING", missing_packages, flush=True)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
    )

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
if importlib.metadata.version("open_clip_torch") != "3.3.0":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "open_clip_torch==3.3.0"])
import open_clip

resolved_commit, translator_assets, translator_assets_sha = provision_translator()
                                                                              
                                                                        
try:
    siglip_wrapper = open_clip.get_tokenizer(
        "ViT-B-16-SigLIP", cache_dir=str(SIGLIP_TOKENIZER), local_files_only=True
    )
except Exception:
    snapshot = (SIGLIP_TOKENIZER / "models--timm--ViT-B-16-SigLIP" / "snapshots"
                / "41f575766f40e752fdd1383e9565b7f02388c1c4")
    required_snapshot_files = ("special_tokens_map.json", "tokenizer.json",
                               "tokenizer_config.json")
    require(all((snapshot / name).is_file() for name in required_snapshot_files),
            f"Incomplete pinned SigLIP tokenizer snapshot: {snapshot}")
    model_cfg = open_clip.get_model_config("ViT-B-16-SigLIP")
    tokenizer_kwargs = model_cfg["text_cfg"].get("tokenizer_kwargs", {})
    siglip_wrapper = open_clip.tokenizer.HFTokenizer(
        str(snapshot), context_length=64,
        clean=tokenizer_kwargs.get("clean", "whitespace"), local_files_only=True
    )
siglip_tokenizer = siglip_wrapper.tokenizer
translator_tokenizer = AutoTokenizer.from_pretrained(TRANSLATOR_ROOT, local_files_only=True,
                                                     use_fast=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
translator = AutoModelForSeq2SeqLM.from_pretrained(
    TRANSLATOR_ROOT, local_files_only=True).to(device)
translator.eval()


def token_count(text):
    ids = siglip_tokenizer(text, add_special_tokens=True, padding=False,
                           truncation=False, return_attention_mask=False)["input_ids"]
    return len(ids)


def chunks(text):
    text = " ".join(text.split())
    if token_count(text) <= PROMPT_CONTEXT_LIMIT:
        return [text]
    output, current = [], ""
    for word in text.split():
        candidate = " ".join(filter(None, (current, word)))
        if token_count(candidate) <= PROMPT_CONTEXT_LIMIT:
            current = candidate
        else:
            require(current, f"Single translated token exceeds context: {word!r}")
            output.append(current)
            current = word
    if current:
        output.append(current)
    return output


missing_records = [test_by_id[query_id] for query_id in sorted(MISSING_IDS)]
segments_by_id = {row["query_id"]: segments(row["query_text_vi"]) for row in missing_records}
unique_segments = list(dict.fromkeys(segment for query_id in sorted(segments_by_id)
                                     for segment in segments_by_id[query_id]))
translations = {}
for offset in range(0, len(unique_segments), 8):
    batch = unique_segments[offset:offset + 8]
    encoded = translator_tokenizer(batch, return_tensors="pt", padding=True,
                                   truncation=False).to(device)
    with torch.no_grad():
        generated = translator.generate(**encoded, num_beams=4, do_sample=False,
                                        max_new_tokens=256, renormalize_logits=True)
    translated = translator_tokenizer.batch_decode(generated, skip_special_tokens=True)
    for source, target in zip(batch, translated, strict=True):
        normalized = " ".join(target.split())
        require(normalized, "Translator returned an empty string")
        translations[source] = normalized
    print("AG32_TRANSLATION_PROGRESS", min(offset + len(batch), len(unique_segments)),
          "/", len(unique_segments), flush=True)

new_records = []
for row in missing_records:
    source_parts = segments_by_id[row["query_id"]]
    translated_parts = [translations[source] for source in source_parts]
    candidates = [("full_translation", " ".join(translated_parts))]
    for index, translated in enumerate(translated_parts, 1):
        candidates.extend((f"segment_{index:02d}_chunk_{chunk_index:02d}", text)
                          for chunk_index, text in enumerate(chunks(translated), 1))
    seen, prompts = set(), []
    for role, text in candidates:
        if text in seen or token_count(text) > PROMPT_CONTEXT_LIMIT:
            continue
        seen.add(text)
        prompts.append({"prompt_role": role, "text_en": text,
                        "text_en_sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(),
                        "siglip_token_count": token_count(text)})
    require(prompts, f"{row['query_id']}: no prompt within the SigLIP context")
    new_records.append({
        "schema_version": 1,
        "query_id": row["query_id"],
        "canonical_query_text_vi_sha256": row["query_text_sha256"],
        "translation_method": "offline_marian_vi_en_beam4",
        "translator_repository": TRANSLATOR_REPOSITORY,
        "translator_resolved_commit": resolved_commit,
        "translator_assets_sha256": translator_assets_sha,
        "source_segments_vi": source_parts,
        "translated_segments_en": translated_parts,
        "retrieval_prompts": prompts,
    })

full_records = []
for query_id in sorted(test_by_id):
    record = old_by_id.get(query_id)
    if record is None:
        record = next(row for row in new_records if row["query_id"] == query_id)
    require(record["canonical_query_text_vi_sha256"] ==
            test_by_id[query_id]["query_text_sha256"], f"{query_id}: prompt binding mismatch")
    full_records.append(record)
require(len(full_records) == 48, "Complete prompt catalog must contain 48 records")

immutable_write(NEW_PROMPTS, b"".join(canon(row) for row in new_records))
immutable_write(FULL_PROMPTS, b"".join(canon(row) for row in full_records))
manifest = {
    "schema_version": 1,
    "status": "test48_prompt_centroid_catalog_locked",
    "artifact_class": "derived_english_retrieval_prompts_not_ground_truth",
    "test48_sha256": sha(TEST),
    "split_lock_sha256": sha(SPLIT_LOCK),
    "source_prompt_catalog_sha256": sha(OLD_PROMPTS),
    "translator_repository": TRANSLATOR_REPOSITORY,
    "translator_resolved_commit": resolved_commit,
    "translator_assets": translator_assets,
    "translator_assets_sha256": translator_assets_sha,
    "translation_method": "offline_marian_vi_en_beam4",
    "prompt_context_limit": PROMPT_CONTEXT_LIMIT,
    "inherited_query_count": 32,
    "new_query_count": 16,
    "query_count": 48,
    "prompt_count": sum(len(row["retrieval_prompts"]) for row in full_records),
    "prompt_catalog_path": str(FULL_PROMPTS),
    "prompt_catalog_sha256": sha(FULL_PROMPTS),
    "canonical_query_text_modified": False,
    "ready_for_test48_region_selection": True,
    "evaluation_executed": False,
    "test_results_observed": False,
}
immutable_write(MANIFEST, canon(manifest))

print("AG32_TEST48_PROMPT_CENTROID_EXPANSION_OK")
print(json.dumps({**manifest, "manifest_path": str(MANIFEST),
                  "manifest_sha256": sha(MANIFEST)}, ensure_ascii=False, indent=2))


In [ ]:
%%stage AG33
                                                                                  
import hashlib,json,os,shutil,subprocess,sys,urllib.request,zipfile,importlib.metadata
from pathlib import Path
import numpy as np, torch
from google.colab import drive,userdata
drive.mount('/content/drive')
os.environ.update(HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',HF_HUB_DISABLE_TELEMETRY='1')
C='ab526c50f491bae440ea697fca72adcb10c687f3'; R=Path('/content/reproduction_inputs/artifacts'); D=Path('/content/reproduction_inputs/dataset')
L=R/'evaluation/round1_evaluation'; S=R/'evaluation/extension_query_promotion'; O=L/'ag33_test48_region_selection_v1'
Q=S/'organizer58_promotion_ready_annotations.jsonl'; T=L/'test48.json'; SL=L/'dev10_test48_split_lock_manifest.json'; PC=L/'prompt_centroid_v1/test48_prompt_catalog.jsonl'; PM=L/'prompt_centroid_v1/test48_prompt_catalog_manifest.json'
CR=R/'frame_cache/model_frame_cache'; CM=CR/'manifest.json'; ER=R/'catalog/expected_video_ids.json'; RR=R/'frame_cache/repaired_frames/L22_V011'; DM=RR/'target_dataset_manifest_L22_V011_046_repair.json'; CK=R/'artifacts/models/ViT-B-16-SigLIP_webli/open_clip_model.safetensors'; TC=R/'artifacts/models/ViT-B-16-SigLIP_webli/tokenizer_cache'
H={Q:'71e43cadb55c884852b31c6297065ba8e59fdeb3ed92b532bcb68c17d2e7a246',T:'231ff1f0432be2bbc1eee9f879a5284f3d6c7633c5c674fdabd605495fd607f5',SL:'c4bac6f9eefdbe072b1ee9f6a44d7fd88df863139967c594fda1798a71c6fa8b',PC:'e944ef5cbdda7c67930f92fc0ee6151fffbfc5d477d63c9886bed2a8a7fda066',PM:'c1ffefa47221fa26bfd976a25a2d6c0568af6e1d4883dadefe5c3f3f9c36227b',ER:'b47517d27e92f5b18d65579964c2cf8ea31454e3853b31aa939f3d2bb9948feb',CM:'5cc4b25610bd433a2abcc38a81dc234dc09bc1f83ecfc4a8dfe8b248e9c7b2a4',DM:'9d3a579384bd8a94ab405a1bfbb39c065705140b593da310176f3f9c7d9351ac',CK:'81942ea8c09b9e41963357cca5d2682118bf7eb7491c2ae51a28bc4f4f95b194'}
MH='1d62e634a2ee5092794b3b3fbfeb07b0d9bdc3727b22d19a96ea2e25e654443a'
def req(x,m):
 if not x: raise RuntimeError(m)
def fh(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
 return h.hexdigest()
def cb(x):return (json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'))+'\n').encode()
def write(p,b):
 p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
 if p.exists():req(p.read_bytes()==b,f'Immutable artifact drift: {p}')
 else:
  z=p.with_suffix(p.suffix+'.tmp');z.write_bytes(b);os.replace(z,p)
for p,h in H.items():req(p.exists() and fh(p)==h,f'Missing/hash drift: {p}')
for p in (D,TC):req(p.exists(),f'Missing: {p}')
lock=json.loads(SL.read_bytes()); pm=json.loads(PM.read_bytes()); rows=json.loads(T.read_bytes())
req(lock.get('locked_evaluation_ready') is True and lock.get('evaluation_executed') is False,'AG30 not ready')
req(pm.get('status')=='test48_prompt_centroid_catalog_locked' and len(rows)==48,'AG32/test48 invalid')
tok=userdata.get('GITHUB_TOKEN');req(isinstance(tok,str) and tok.strip(),'Add GITHUB_TOKEN to Colab Secrets')
rt=Path('/content/stv_ag33_runtime');zp=Path('/content/stv_ag33.zip');shutil.rmtree(rt,ignore_errors=True);rt.mkdir()
rq=urllib.request.Request(f'https://api.github.com/repos/haruxne/STV-Attack/zipball/{C}',headers={'Authorization':f'Bearer {tok.strip()}','Accept':'application/vnd.github+json'})
with urllib.request.urlopen(rq,timeout=120) as x:zp.write_bytes(x.read())
with zipfile.ZipFile(zp) as z:z.extractall(rt)
roots=[p for p in rt.iterdir() if p.is_dir()];req(len(roots)==1,'Bad repository archive');sys.path.insert(0,str(roots[0]))
for n in list(sys.modules):
 if n=='src' or n.startswith('src.'):del sys.modules[n]
try:v=importlib.metadata.version('open_clip_torch')
except importlib.metadata.PackageNotFoundError:v=None
if v!='3.3.0':subprocess.check_call([sys.executable,'-m','pip','install','-q','open_clip_torch==3.3.0'])
from src.clip_windows import build_sliding_windows,target_centered_window
from src.embedding_cache import _load_and_validate_matrix
from src.experiment_runner import OpenClipSiglipAdapter,encode_query,load_embedding_cache,load_expected_video_ids,load_query_prompts,model_identity_sha256,canonical_model_identity
from src.retrieval import VideoFrameEmbeddings,_l2_normalize_rows,prepare_video
cfg={'window_size':16,'stride':1,'query_text_field':'query_text_en','query_representation':{'mode':'english_prompt_centroid','catalog_path':str(PC),'catalog_sha256':H[PC]}}
prompts=load_query_prompts(cfg,rows);ids,_=load_expected_video_ids(ER,expected_sha256=H[ER]);print('AG33_LOADING_CACHE',flush=True)
cache=load_embedding_cache(CM,cache_root=CR,expected_manifest_sha256=H[CM],expected_dataset_hash=H[DM],expected_model_hash=MH,expected_video_ids=ids)
adapter=OpenClipSiglipAdapter.load_local(checkpoint_path=CK,tokenizer_cache_dir=TC,device='cpu',expected_checkpoint_sha256=H[CK]);req(model_identity_sha256(canonical_model_identity(adapter))==MH,'Model drift')
proto={'schema_version':1,'scope':'locked_test48_cache_geometry_selection_not_attack_success','source_commit':C,'budget':16,'contiguous_only':True,'query_sha256':H[Q],'test_sha256':H[T],'split_lock_sha256':H[SL],'cache_sha256':H[CM],'prompt_sha256':H[PC],'prompt_manifest_sha256':H[PM],'model_sha256':MH,'evaluation_executed':False,'test_results_observed':False}
PP=O/'protocol.json';write(PP,cb(proto));ph=fh(PP);by={x.video_id:x for x in cache.records};reports=[]
def analysis(scores,windows,n,target):
 def ev(s):
  u=[i for i,w in enumerate(windows) if w[-1]<s or w[0]>=s+16];return {'start':s,'positions':list(range(s,s+16)),'unaffected_score_floor':max((float(scores[i]) for i in u),default=None),'unaffected_count':len(u)}
 choices=[ev(s) for s in range(n-15)];best=min(choices,key=lambda x:(float('-inf') if x['unaffected_score_floor'] is None else x['unaffected_score_floor'],x['start']));top=max(range(len(scores)),key=lambda i:float(scores[i]));ts=min(target);req(list(target)==list(range(ts,ts+16)),'Bad target interval')
 return {'scope':'contiguous_intervals_only_same_16_frame_budget','target_centered':ev(ts),'highest_score_window':ev(min(windows[top][0],n-16)),'minimum_unaffected_floor':best,'intervals_evaluated':len(choices)}
for r in rows:
 q=r['query_id'];dst=O/(q+'.json')
 if dst.exists():x=json.loads(dst.read_bytes());req(x['protocol_sha256']==ph,'Resume drift');reports.append(x);print('AG33_RESUMED',q,flush=True);continue
 print('AG33_QUERY',q,flush=True);req(r['video_id'] in by,f'{q}: video absent from cache');m=_load_and_validate_matrix(cache.root,by[r['video_id']]);f=torch.from_numpy(np.array(m,copy=True));w=build_sliding_windows(len(f),window_size=16,stride=1);pv=prepare_video(VideoFrameEmbeddings(video_id=r['video_id'],clips=tuple(f[list(a)] for a in w)))
 with torch.inference_mode():qe=encode_query(adapter,prompts[q],centroid=True).cpu();e=_l2_normalize_rows(pv.embeddings,name='cached clips');qe=_l2_normalize_rows(qe,name='query').to(e);scores=torch.mv(e,qe).tolist()
 target=sorted(set(target_centered_window(len(f),r['target_keyframe_n']-1,window_size=16)));x={'query_id':q,'video_id':r['video_id'],'clean_score':max(scores),'protocol_sha256':ph,'analysis':analysis(scores,w,len(f),target)};write(dst,cb(x));reports.append(x)
req(len(reports)==48 and len({x['query_id'] for x in reports})==48,'Incomplete test48 selection');CP=O/'comparison.json';write(CP,cb({'protocol':proto,'results':sorted(reports,key=lambda x:x['query_id'])}))
print('AG33_TEST48_REGION_SELECTION_COMPLETE');print(json.dumps({'comparison_path':str(CP),'comparison_sha256':fh(CP),'query_count':48,'evaluation_executed':False,'test_results_observed':False},indent=2))


In [ ]:
%%stage AG34
                                                                                
import hashlib,json,os
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
R=Path('/content/reproduction_inputs/artifacts'); OLD=R/'evaluation/round1_method_lock'; L=R/'evaluation/round1_evaluation'
METHOD=OLD/'ag19_region_comparison_v1/region_method_lock_v1.json'; SPLIT=L/'dev10_test48_split_lock_manifest.json'; TEST=L/'test48.json'; PM=L/'prompt_centroid_v1/test48_prompt_catalog_manifest.json'; PC=L/'prompt_centroid_v1/test48_prompt_catalog.jsonl'; SEL=L/'ag33_test48_region_selection_v1/comparison.json'; OUT=L/'ag34_test48_final_pre_evaluation_lock.json'
H={METHOD:'8dc2e1e255f20f3e1289bca8bfe5fe4e79e3863e1d80710997aab621089d10bb',SPLIT:'c4bac6f9eefdbe072b1ee9f6a44d7fd88df863139967c594fda1798a71c6fa8b',TEST:'231ff1f0432be2bbc1eee9f879a5284f3d6c7633c5c674fdabd605495fd607f5',PM:'c1ffefa47221fa26bfd976a25a2d6c0568af6e1d4883dadefe5c3f3f9c36227b',PC:'e944ef5cbdda7c67930f92fc0ee6151fffbfc5d477d63c9886bed2a8a7fda066',SEL:'8aa3c5854e32aade4947fed14886a776823974646e0bb5c4cf3ce67d468c88ac'}
def req(x,m):
 if not x:raise RuntimeError(m)
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def canon(x):return (json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'))+'\n').encode()
for p,h in H.items():req(p.is_file() and sha(p)==h,f'Missing/hash drift: {p}')
m=json.loads(METHOD.read_bytes());s=json.loads(SPLIT.read_bytes());pm=json.loads(PM.read_bytes());sel=json.loads(SEL.read_bytes());test=json.loads(TEST.read_bytes())
req(m.get('status')=='method_frozen_pending_evaluation_readiness' and m.get('selected_method')=='minimum_unaffected_floor','Dev method lock invalid')
req(m.get('baselines')==['target_centered','highest_score_window'],'Baseline lock invalid')
req(s.get('locked_evaluation_ready') is True and s.get('evaluation_executed') is False,'Split not ready')
req(pm.get('status')=='test48_prompt_centroid_catalog_locked' and pm.get('query_count')==48,'Prompt lock invalid')
req(sel.get('protocol',{}).get('scope')=='locked_test48_cache_geometry_selection_not_attack_success','AG33 scope invalid')
req(len(sel.get('results',[]))==len(test)==48,'AG33/test48 count mismatch')
ids={r['query_id'] for r in test};req({r['query_id'] for r in sel['results']}==ids,'AG33 query set mismatch')
strategies=['target_centered','highest_score_window','minimum_unaffected_floor']
for r in sel['results']:
 for name in strategies:
  pos=r['analysis'][name]['positions'];req(len(pos)==16 and pos==list(range(pos[0],pos[0]+16)),f"{r['query_id']}: invalid {name} interval")
lock={'schema_version':1,'status':'test48_final_pre_evaluation_configuration_locked','artifact_class':'pretest_lock_for_promoted_region_selection_evaluation','method_lock_sha256':H[METHOD],'split_lock_sha256':H[SPLIT],'test48_sha256':H[TEST],'prompt_catalog_sha256':H[PC],'prompt_manifest_sha256':H[PM],'region_selection_sha256':H[SEL],'selected_method':'minimum_unaffected_floor','baselines':['target_centered','highest_score_window'],'strategies':strategies,'query_count':48,'video_count':46,'region_budget':16,'region_contiguous':True,'epsilon':8/255,'alpha_start':2/255,'alpha_end':0.2/255,'steps':15,'lambda1':0.5,'lambda2':0.0,'random_seed':42,'window_size':16,'stride':1,'query_representation':'english_prompt_centroid','uses_mock':False,'locked_evaluation_ready':True,'evaluation_executed':False,'test_results_observed':False,'evaluation_rule':'Run all three locked strategies once on test-48; report every failure and paired denominator; no tuning after observation.'}
b=canon(lock);OUT.parent.mkdir(parents=True,exist_ok=True)
if OUT.exists():req(OUT.read_bytes()==b,'Immutable AG34 lock drift')
else:
 t=OUT.with_suffix('.tmp');t.write_bytes(b);os.replace(t,OUT)
print('AG34_TEST48_FINAL_PRE_EVALUATION_LOCK_OK')
print(json.dumps({**lock,'path':str(OUT),'sha256':sha(OUT)},ensure_ascii=False,indent=2))


In [ ]:
%%stage AG35
                  
                                                                                
                                                                                        
import gc, hashlib, importlib.metadata, json, os, random, shutil, subprocess, sys, time, urllib.request, zipfile
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def require(value, message):
    if not value:
        raise RuntimeError(message)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def canonical_bytes(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True,
                       separators=(",", ":")) + "\n").encode("utf-8")

def atomic_immutable(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        require(path.read_bytes() == payload, f"Immutable artifact drift: {path}")
        return
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, path)

STV_ROOT = Path("/content/reproduction_inputs/artifacts")
DATA_ROOT = Path("/content/reproduction_inputs/dataset")
PILOT_ROOT = STV_ROOT / "evaluation/round1_evaluation"
QUERIES = STV_ROOT / "evaluation/extension_query_promotion/organizer58_promotion_ready_annotations.jsonl"
FREEZE_MANIFEST = STV_ROOT / "evaluation/extension_query_promotion/organizer58_promotion_ready_manifest.json"
DEV = PILOT_ROOT / "dev10.json"
TEST = PILOT_ROOT / "test48.json"
SPLIT_MANIFEST = PILOT_ROOT / "dev10_test48_split_lock_manifest.json"
PROMPT_CATALOG = PILOT_ROOT / "prompt_centroid_v1/test48_prompt_catalog.jsonl"
RECOVERY_ROOT = STV_ROOT / "frame_cache/repaired_frames/L22_V011"
CACHE_ROOT = STV_ROOT / "frame_cache/model_frame_cache"
OFFICIAL_DATASET_MANIFEST = RECOVERY_ROOT / "target_dataset_manifest_L22_V011_046_repair.json"
EXPECTED_VIDEO_IDS = STV_ROOT / "catalog/expected_video_ids.json"
CACHE_MANIFEST = CACHE_ROOT / "manifest.json"
CHECKPOINT = STV_ROOT / "artifacts/models/ViT-B-16-SigLIP_webli/open_clip_model.safetensors"
TOKENIZER_CACHE = STV_ROOT / "artifacts/models/ViT-B-16-SigLIP_webli/tokenizer_cache"
OUTPUT_ROOT = PILOT_ROOT / "ag35_final_test48_region_evaluation_v5"
CONFIG_PATH = OUTPUT_ROOT / "centroid_t16_config.json"
AUTHORITY_PATH = OUTPUT_ROOT / "dev10_centroid_authority.jsonl"
SUMMARY_PATH = OUTPUT_ROOT / "dev10_centroid_summary.json"
STATISTICS_PATH = OUTPUT_ROOT / "dev10_centroid_statistics.json"
COMPLETION_PATH = OUTPUT_ROOT / "ag10_completion.json"

EXPECTED_QUERY_SHA = "71e43cadb55c884852b31c6297065ba8e59fdeb3ed92b532bcb68c17d2e7a246"
EXPECTED_DEV_SHA = "278a15fdc6b8aee35b1336b52206194f63d60143040b70090eea3ba2664eedac"
EXPECTED_TEST_SHA = "231ff1f0432be2bbc1eee9f879a5284f3d6c7633c5c674fdabd605495fd607f5"
EXPECTED_PROMPT_SHA = "e944ef5cbdda7c67930f92fc0ee6151fffbfc5d477d63c9886bed2a8a7fda066"
EXPECTED_DATASET_SHA = "9d3a579384bd8a94ab405a1bfbb39c065705140b593da310176f3f9c7d9351ac"
EXPECTED_VIDEO_IDS_SHA = "b47517d27e92f5b18d65579964c2cf8ea31454e3853b31aa939f3d2bb9948feb"
EXPECTED_CACHE_SHA = "5cc4b25610bd433a2abcc38a81dc234dc09bc1f83ecfc4a8dfe8b248e9c7b2a4"
EXPECTED_MODEL_SHA = "1d62e634a2ee5092794b3b3fbfeb07b0d9bdc3727b22d19a96ea2e25e654443a"
EXPECTED_CHECKPOINT_SHA = "81942ea8c09b9e41963357cca5d2682118bf7eb7491c2ae51a28bc4f4f95b194"

required_paths = [QUERIES, FREEZE_MANIFEST, DEV, SPLIT_MANIFEST, PROMPT_CATALOG,
                  DATA_ROOT, OFFICIAL_DATASET_MANIFEST, EXPECTED_VIDEO_IDS,
                  CACHE_MANIFEST, CACHE_ROOT, CHECKPOINT, TOKENIZER_CACHE]
require(not [str(path) for path in required_paths if not path.exists()],
        f"Missing required assets: {[str(path) for path in required_paths if not path.exists()]}")
for path, expected, label in (
    (QUERIES, EXPECTED_QUERY_SHA, "accepted37"),
    (TEST, EXPECTED_TEST_SHA, "test-48 split"),
    (PROMPT_CATALOG, EXPECTED_PROMPT_SHA, "prompt catalog"),
    (OFFICIAL_DATASET_MANIFEST, EXPECTED_DATASET_SHA, "dataset manifest"),
    (EXPECTED_VIDEO_IDS, EXPECTED_VIDEO_IDS_SHA, "expected video IDs"),
    (CACHE_MANIFEST, EXPECTED_CACHE_SHA, "cache manifest"),
    (CHECKPOINT, EXPECTED_CHECKPOINT_SHA, "checkpoint"),
):
    require(file_hash(path) == expected, f"{label} SHA-256 mismatch")

freeze = json.loads(FREEZE_MANIFEST.read_bytes())
split = json.loads(SPLIT_MANIFEST.read_bytes())
require(freeze.get("status") == "organizer58_promotion_ready_pending_split_lock", "AG29 status drift")
require(freeze.get("merged_promoted", {}).get("sha256") == EXPECTED_QUERY_SHA, "AG29 hash drift")
require(split.get("status") == "organizer58_dev10_test48_locked_before_evaluation", "AG30 status drift")
require(split.get("test", {}).get("sha256") == EXPECTED_TEST_SHA and split.get("test", {}).get("query_count") == 48, "AG30 test48 drift")
require(split.get("locked_evaluation_ready") is True and split.get("evaluation_executed") is False, "AG30 not ready")

try:
    version = importlib.metadata.version("open_clip_torch")
except importlib.metadata.PackageNotFoundError:
    version = None
if version != "3.3.0":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch==3.3.0"])

RUNTIME_ROOT = Path("/content/stv_ag35_runtime")
runtime_zip = Path("/content/stv_ag35_runtime.zip")
token = userdata.get("GITHUB_TOKEN")
require(isinstance(token, str) and token.strip(), "Add GITHUB_TOKEN to Colab Secrets")
if RUNTIME_ROOT.exists(): shutil.rmtree(RUNTIME_ROOT)
RUNTIME_ROOT.mkdir(parents=True)
request = urllib.request.Request(
    "https://api.github.com/repos/haruxne/STV-Attack/zipball/ab526c50f491bae440ea697fca72adcb10c687f3",
    headers={"Authorization": f"Bearer {token.strip()}", "Accept": "application/vnd.github+json"})
with urllib.request.urlopen(request, timeout=120) as response:
    runtime_zip.write_bytes(response.read())
with zipfile.ZipFile(runtime_zip) as archive: archive.extractall(RUNTIME_ROOT)
roots = [path for path in RUNTIME_ROOT.iterdir() if path.is_dir()]
require(len(roots) == 1 and (roots[0] / "src").is_dir(), "Unexpected repository archive")
RUNTIME_ROOT = roots[0]
sys.path.insert(0, str(RUNTIME_ROOT))
for _name in list(sys.modules):
    if _name == "src" or _name.startswith("src."): del sys.modules[_name]
import torch
from src.clean_runner import CleanConfig, _load_and_validate_config as load_clean_config, run_clean_retrieval
from src.experiment_runner import AttackConfig, _load_and_validate_config as load_attack_config, run_attack_experiment
from src.result_authority import load_result_authority
from src.statistics import build_statistical_report

require(torch.cuda.is_available(), "AG10 requires CUDA")
print("AG35_GPU", torch.cuda.get_device_name(0), flush=True)
gc.collect()
torch.cuda.empty_cache()
free_gpu, total_gpu = torch.cuda.mem_get_info()
print("AG35_GPU_FREE_GIB", round(free_gpu / 2**30, 2), "OF", round(total_gpu / 2**30, 2), flush=True)
require(
    free_gpu >= 18 * 2**30,
    "GPU already contains live tensors/models. Restart the Colab session, then run only this AG35 cell.",
)

window_config = {
    "window_size": 16,
    "stride": 1,
    "query_text_field": "query_text_en",
    "query_representation": {
        "mode": "english_prompt_centroid",
        "catalog_path": str(PROMPT_CATALOG),
        "catalog_sha256": EXPECTED_PROMPT_SHA,
    },
}
atomic_immutable(CONFIG_PATH, canonical_bytes(window_config))
_, clean_hash = load_clean_config(CONFIG_PATH)
common = dict(
    queries=QUERIES, expected_query_sha256=EXPECTED_QUERY_SHA,
    split=DEV, expected_split_sha256=EXPECTED_DEV_SHA,
    cache_manifest=CACHE_MANIFEST, expected_cache_manifest_sha256=EXPECTED_CACHE_SHA,
    cache_root=CACHE_ROOT, data_root=DATA_ROOT,
    dataset_hash=EXPECTED_DATASET_SHA, model_hash=EXPECTED_MODEL_SHA,
    checkpoint_path=CHECKPOINT, expected_checkpoint_sha256=EXPECTED_CHECKPOINT_SHA,
    tokenizer_cache_dir=TOKENIZER_CACHE, config=CONFIG_PATH,
    authority_output=AUTHORITY_PATH, device="cuda", resume=True,
    expected_video_ids=EXPECTED_VIDEO_IDS, expected_video_ids_sha256=EXPECTED_VIDEO_IDS_SHA,
    official_dataset_manifest=OFFICIAL_DATASET_MANIFEST,
)



import inspect
import random
import numpy as np
import src.experiment_runner as runner

                                                                           
                                                                             
                                                                          
                                                                          
promoted_by_id = {r['query_id']: r for r in map(json.loads, QUERIES.read_text(encoding='utf-8-sig').splitlines()) if r}
require(len(promoted_by_id)==58, 'Organizer58 promotion input is incomplete')
def validate_organizer_execution_annotations(records, *, data_root,
        official_dataset_manifest, expected_manifest_sha256, locked=True):
    errors=[]
    if locked is not True:
        errors.append({'line':0,'field':'locked','message':'organizer evaluation requires locked=True'})
    if file_hash(official_dataset_manifest)!=expected_manifest_sha256:
        errors.append({'line':0,'field':'official_dataset_manifest','message':'manifest hash drift'})
    manifest=json.loads(official_dataset_manifest.read_bytes())
    videos={v['video_id']:v for v in manifest.get('videos',[])}
    seen=set()
    for line,row in enumerate(records,1):
        qid=row.get('query_id'); source=promoted_by_id.get(qid)
        if source is None or qid in seen:
            errors.append({'line':line,'field':'query_id','message':'missing/duplicate promoted organizer ID'});continue
        seen.add(qid)
        for field in ('query_text_vi','query_text_sha256','video_id','target_keyframe_n','target_frame_id','keyframe_sha256','mapping_sha256'):
            if row.get(field)!=source.get(field):errors.append({'line':line,'field':field,'message':'differs from hash-bound promotion source'})
        if source.get('ground_truth_promoted') is not True or source.get('ground_truth_inferred') is True:
            errors.append({'line':line,'field':'ground_truth_promoted','message':'target is not human-promoted'})
        video=videos.get(row.get('video_id'))
        if video is None:errors.append({'line':line,'field':'video_id','message':'absent from official manifest'})
    return {'schema_version':1,'status':'valid' if not errors else 'invalid','locked':True,
            'record_count':len(records),'distinct_video_count':len({r.get('video_id') for r in records}),
            'validator':'organizer_promotion_hash_binding_v1','errors':errors,'warnings':[]}

require(Path(inspect.getfile(runner)).resolve().is_relative_to(RUNTIME_ROOT.resolve()),
        "Unexpected runner import")
selected = sorted(json.loads(TEST.read_bytes()), key=lambda r: r["query_id"] )
split_path = OUTPUT_ROOT / "all_test48.json"
atomic_immutable(split_path, canonical_bytes(selected))
common.update(split=split_path, expected_split_sha256=file_hash(split_path))

original = runner.run_pgd_attack

def memory_safe_scheduled_attack(adapter, video_frames, query_embedding, config,
                                 window_size, stride, changed_positions=None):
    """Exact locked PGD objective with only the editable interval resident on CUDA."""
    device = torch.device(config.device)
    require(device.type == "cuda", "AG35 memory-safe runner requires CUDA")
    require(query_embedding.ndim == 1, "query_embedding must be one-dimensional")
    query = query_embedding.detach().to(device)
    frame_count = int(video_frames.shape[0])
    all_windows = runner.build_sliding_windows(frame_count, window_size=window_size, stride=stride)
    changed = tuple(range(frame_count)) if changed_positions is None else tuple(sorted(set(changed_positions)))
    require(changed and all(isinstance(p, int) and not isinstance(p, bool) and 0 <= p < frame_count for p in changed),
            "changed_positions must identify original video frames")
    require(changed == tuple(range(changed[0], changed[0] + len(changed))),
            "AG35 requires one contiguous editable interval")
    affected_indices = runner.overlapping_window_indices(
        frame_count, changed, window_size=window_size, stride=stride)
    windows = tuple(all_windows[i] for i in affected_indices)
    changed_lookup = {position: offset for offset, position in enumerate(changed)}
    clean_changed = video_frames[list(changed)].detach().to(device)

    def temporal_losses_exact(delta):
        pixels = delta.shape[1] * delta.shape[2] * delta.shape[3]
        zero = delta.new_zeros(())
        first_sum = (delta[1:] - delta[:-1]).square().sum() if len(changed) > 1 else zero
        if changed[0] > 0:
            first_sum = first_sum + delta[0].square().sum()
        if changed[-1] < frame_count - 1:
            first_sum = first_sum + delta[-1].square().sum()
        flicker = first_sum / max((frame_count - 1) * pixels, 1)

        accel_sum = zero
        if frame_count > 2:
            for center in range(max(0, changed[0] - 2), min(frame_count - 3, changed[-1]) + 1):
                terms = []
                for position, coefficient in ((center, 1.0), (center + 1, -2.0), (center + 2, 1.0)):
                    offset = changed_lookup.get(position)
                    if offset is not None:
                        terms.append(coefficient * delta[offset])
                if terms:
                    curvature = terms[0]
                    for term in terms[1:]:
                        curvature = curvature + term
                    accel_sum = accel_sum + curvature.square().sum()
        acceleration = accel_sum / max((frame_count - 2) * pixels, 1)
        return flicker, acceleration

    def clip_for(window, delta):
        frames = []
        for position in window:
            offset = changed_lookup.get(position)
            if offset is None:
                frames.append(video_frames[position].to(device))
            else:
                frames.append(torch.clamp(clean_changed[offset] + delta[offset], 0.0, 1.0))
        return torch.stack(frames).unsqueeze(0)

    def streaming_max_similarity(delta, *, gradients):
        context = torch.enable_grad() if gradients else torch.no_grad()
        best = None
        with context:
            for window in windows:
                clip = clip_for(window, delta)
                score = torch.mv(adapter.get_video_embedding(clip), query).squeeze(0)
                if best is None or float(score.detach()) > float(best.detach()):
                    best = score
                del clip
        require(best is not None, "No affected retrieval window")
        return best

    if config.condition == "random":
        generator = torch.Generator(device=device)
        generator.manual_seed(config.random_seed)
        delta = torch.empty_like(clean_changed).uniform_(-config.epsilon, config.epsilon,
                                                          generator=generator)
        delta = torch.maximum(torch.minimum(delta, 1.0 - clean_changed), -clean_changed)
        iterations = 1
    else:
        delta = torch.zeros_like(clean_changed, requires_grad=True)
        for step in range(config.steps):
            sim_loss = streaming_max_similarity(delta, gradients=True)
            flicker_loss, accel_loss = temporal_losses_exact(delta)
            loss = sim_loss + config.lambda1 * flicker_loss + config.lambda2 * accel_loss
            loss.backward()
            require(delta.grad is not None, "PGD produced no perturbation gradient")
            alpha = config.alpha * (1.0 - 0.9 * step / max(config.steps - 1, 1))
            with torch.no_grad():
                delta.sub_(alpha * delta.grad.sign())
                delta.clamp_(-config.epsilon, config.epsilon)
                delta.copy_(torch.maximum(torch.minimum(delta, 1.0 - clean_changed), -clean_changed))
            delta.grad = None
        iterations = config.steps

    with torch.no_grad():
        adversarial_changed = torch.clamp(clean_changed + delta.detach(), 0.0, 1.0)
        sim_loss = streaming_max_similarity(delta.detach(), gradients=False)
        flicker_loss, accel_loss = temporal_losses_exact(delta.detach())
        squared_error = (adversarial_changed - clean_changed).square().sum().item()
        mse = squared_error / video_frames.numel()
        psnr = float("inf") if mse == 0 else 10.0 * __import__("math").log10(1.0 / mse)
        changed_ssim = sum(
            runner.compute_ssim(clean_changed[i:i + 1], adversarial_changed[i:i + 1])
            for i in range(len(changed))
        ) / len(changed)
        ssim = ((frame_count - len(changed)) + len(changed) * changed_ssim) / frame_count
        linf = delta.detach().abs().max().item()
        adversarial_cpu = adversarial_changed.cpu()

    # Reuse the already-loaded CPU tensor; only the locked 16-frame interval changes.
    video_frames[list(changed)] = adversarial_cpu
    metrics = {
        "psnr": psnr if __import__("math").isfinite(psnr) else None,
        "psnr_reason": "identical tensors" if not __import__("math").isfinite(psnr) else None,
        "ssim": float(ssim), "perturbation_linf": linf,
        "flicker": flicker_loss.item(), "acceleration": accel_loss.item(),
        "loss_sim": sim_loss.item(), "iterations": iterations,
        "changed_frame_count": len(changed),
        "affected_window_count": len(affected_indices),
    }
    del clean_changed, delta, adversarial_changed, adversarial_cpu, query
    torch.cuda.empty_cache()
    return video_frames, metrics

scheduled = memory_safe_scheduled_attack
scheduled_source = inspect.getsource(memory_safe_scheduled_attack)

require(len(selected)==48, 'Expected locked test48')
ag18_root=PILOT_ROOT/'ag33_test48_region_selection_v1'
ag18_path=ag18_root/'comparison.json'
ag23_path=PILOT_ROOT/'ag34_test48_final_pre_evaluation_lock.json'
require(ag23_path.is_file(), 'Finish AG23 final pre-test lock first')
ag23=json.loads(ag23_path.read_bytes())
require(file_hash(ag23_path)=='b0bb64a6068fe4151a5a606aafedc6f6b1413ae04f6c69b368bce462356d5e4a', 'AG34 lock hash drift')
require(ag23.get('status')=='test48_final_pre_evaluation_configuration_locked' and ag23.get('locked_evaluation_ready') is True, 'AG23 lock is not evaluation-ready')
require(ag23.get('test48_sha256')==EXPECTED_TEST_SHA and ag23.get('query_count')==48, 'AG23 test lock mismatch')
require(ag18_path.is_file(), 'Finish AG18 first')
ag18=json.loads(ag18_path.read_bytes())
require(file_hash(ag18_path)=='8aa3c5854e32aade4947fed14886a776823974646e0bb5c4cf3ce67d468c88ac', 'AG33 comparison hash drift')
require(ag18['protocol']['test_sha256']==EXPECTED_TEST_SHA and
        ag18['protocol']['cache_sha256']==EXPECTED_CACHE_SHA and
        ag18['protocol']['prompt_sha256']==EXPECTED_PROMPT_SHA and
        ag18['protocol']['budget']==16 and ag18['protocol']['contiguous_only'] is True,
        'AG33 input/protocol mismatch')
selection_by_query={r['query_id']:r for r in ag18['results']}
require(len(selection_by_query)==48 and set(selection_by_query)=={r['query_id'] for r in selected},
        'AG18 query set mismatch')
strategies=('target_centered','highest_score_window','minimum_unaffected_floor')
for record in selected:
    saved=selection_by_query[record['query_id']]
    require(saved['video_id']==record['video_id'], 'AG18 target mismatch')
    for strategy in strategies:
        positions=saved['analysis'][strategy]['positions']
        require(len(positions)==16 and positions==list(range(positions[0],positions[0]+16)),
                'Invalid contiguous 16-frame selection')

                                                                              
                                                                    
original_run=runner.run_attack_experiment
run_source=inspect.getsource(original_run)
                                                                           
                                                                              
legacy_keyframe='target_keyframe_n = record.get("keyframe_n")'
require(run_source.count(legacy_keyframe)==1, 'Pinned runner keyframe marker changed')
run_source=run_source.replace(
    legacy_keyframe,
    'target_keyframe_n = record.get("target_keyframe_n")',
)
marker='            # 4. Execute attack'
require(run_source.count(marker)==1, 'Runner source changed')
insertion="""        # AG35: frozen cache-derived selection, same contiguous frame budget.
        chosen = region_selections[query_id]["analysis"][region_strategy]["positions"]
        if region_strategy == "target_centered":
            if list(changed_positions) != chosen:
                raise ExperimentRunnerError("AG24 target-centered baseline drift")
        changed_positions = tuple(chosen)
        if any(p < 0 or p >= video_frames.shape[0] for p in changed_positions):
            raise ExperimentRunnerError("Region outside target video")

"""
insertion='\n'.join('    '+line if line else line for line in insertion.split('\n'))
modified_source=run_source.replace(marker,insertion+marker)
oom_handler='''        except Exception as e:
            logger.error(f"Failure on {query_id}: {e}")'''
require(modified_source.count(oom_handler)==1, 'Runner exception handler changed')
modified_source=modified_source.replace(oom_handler, '''        except Exception as e:
            if isinstance(e, torch.OutOfMemoryError):
                raise
            logger.error(f"Failure on {query_id}: {e}")''')
protocol={'scope':'locked_test48_final_region_evaluation_once_only',
    'test_split_touched':True,'strategies':strategies,'ag33_comparison_sha256':file_hash(ag18_path), 'ag23_lock_sha256':file_hash(ag23_path),
    'query_sha256':EXPECTED_QUERY_SHA,'test_sha256':EXPECTED_TEST_SHA,
    'organizer_validator_sha256':hashlib.sha256(inspect.getsource(validate_organizer_execution_annotations).encode()).hexdigest(),
    'scheduled_function_sha256':hashlib.sha256(scheduled_source.encode()).hexdigest(),
    'region_runner_sha256':hashlib.sha256(modified_source.encode()).hexdigest(),
    'epsilon':8/255,'steps':15,'lambda1':0.5,'lambda2':0.,
    'schedule':'linear alpha 2/255 to 0.2/255', 'frame_budget':16,
    'attack_memory_plan':'streaming exact max-gradient; only the locked editable interval resides persistently on CUDA',
    'threat_model':'attacker may select any single contiguous interval of 16 sampled frames',
    'identity':'protocol hash and strategy accompany original config hash; config hash alone omits selector/schedule',
    'reporting':'All 48 paired ranks, scores, distortion and runtime; compare to highest-score baseline, not just target-centered.'}
atomic_immutable(OUTPUT_ROOT/'protocol.json',canonical_bytes(protocol))
protocol_hash=file_hash(OUTPUT_ROOT/'protocol.json')
print('AG35_START',protocol,flush=True)
reports=[]
try:
    for strategy in strategies:
        namespace=dict(runner.__dict__)
        namespace.update(run_pgd_attack=scheduled, region_strategy=strategy,
                         region_selections=selection_by_query,
                         validate_annotations=validate_organizer_execution_annotations)
        exec(compile(modified_source,'<ag19-region-runner>','exec'),namespace)
        folder=OUTPUT_ROOT/strategy
        identity={'protocol_sha256':protocol_hash,'strategy':strategy,
                  'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0)}
        atomic_immutable(folder/'run_identity.json',canonical_bytes(identity))
        random.seed(42)
        np.random.seed(42)
        torch.manual_seed(42)
        torch.cuda.manual_seed_all(42)
        torch.backends.cudnn.benchmark=False
        path=folder/'authority.jsonl'
        print('AG35_STRATEGY_START',strategy,flush=True)
        namespace['run_attack_experiment'](AttackConfig(**{**common,'authority_output':path},
            condition='first_order',epsilon=8/255,alpha=2/255,steps=15,lambda1=0.5,lambda2=0.,random_seed=42))
        rows=load_result_authority(path)
        require(len(rows)==48 and {r['query_id'] for r in rows}==set(selection_by_query)
                and all(r['status']=='success' for r in rows),'Incomplete/failed strategy; rerun to resume')
        for r in rows:
            require(r['measurements']['changed_positions_zero_based']==
                    selection_by_query[r['query_id']]['analysis'][strategy]['positions'],
                    'Recorded edit positions mismatch')
        report={**identity,'authority_sha256':file_hash(path),'rows':rows}
        reports.append(report)
        print('AG35_STRATEGY_COMPLETE',strategy,flush=True)
        gc.collect()
        torch.cuda.empty_cache()
finally:
    runner.run_pgd_attack=original
by_strategy={r['strategy']:{x['query_id']:x for x in r['rows']} for r in reports}
paired=[]
for qid in sorted(selection_by_query):
    row={'query_id':qid}
    for strategy in strategies:
        r=by_strategy[strategy][qid]
        row[strategy]={'rank':r['rank'], **{k:r['measurements'][k] for k in
            ('score','clean_rank','psnr','ssim','flicker','acceleration','elapsed_time')}}
    paired.append(row)
summary={}
for baseline in strategies[:2]:
    differences=[r['minimum_unaffected_floor']['rank']-r[baseline]['rank'] for r in paired]
    summary[baseline]={'rank_better_equal_worse':[sum(d>0 for d in differences),
        sum(d==0 for d in differences),sum(d<0 for d in differences)],
        'mean_rank_difference':sum(differences)/48}
atomic_immutable(OUTPUT_ROOT/'comparison.json',canonical_bytes(
    {'protocol':protocol,'results':reports,'paired':paired,'summary':summary}))
print('AG35_PAIRED',json.dumps(paired),flush=True)
print('AG35_SUMMARY',json.dumps(summary),flush=True)
print('AG35_COMPLETE',str(OUTPUT_ROOT/'comparison.json'),flush=True)


In [ ]:
%%stage AG36
"""AG36: CPU-only integrity audit and paired statistics for AG35 test-48.

Run only after AG35 prints AG35_COMPLETE. This cell performs no inference and
does not modify the locked split, prompts, selections, or AG35 authority files.
"""
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path
import hashlib
import json
import math
import os

import numpy as np


def require(ok, message):
    if not ok:
        raise RuntimeError(message)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_bytes(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True,
                       separators=(",", ":")) + "\n").encode("utf-8")


def finite_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)


def exact_two_sided_sign_test(wins, losses):
    non_ties = wins + losses
    if non_ties == 0:
        return None
    tail = min(wins, losses)
    probability = 2.0 * sum(math.comb(non_ties, k) for k in range(tail + 1)) / (2 ** non_ties)
    return min(1.0, probability)


ROOT = Path("/content/reproduction_inputs/artifacts")
LOCK_ROOT = ROOT / "evaluation/round1_evaluation"
RUN = LOCK_ROOT / "ag35_final_test48_region_evaluation_v5"
TEST = LOCK_ROOT / "test48.json"
SPLIT_LOCK = LOCK_ROOT / "dev10_test48_split_lock_manifest.json"
PROMPT_CATALOG = LOCK_ROOT / "prompt_centroid_v1/test48_prompt_catalog.jsonl"
SELECTION = LOCK_ROOT / "ag33_test48_region_selection_v1/comparison.json"
PRETEST_LOCK = LOCK_ROOT / "ag34_test48_final_pre_evaluation_lock.json"
PROTOCOL = RUN / "protocol.json"
COMPARISON = RUN / "comparison.json"
OUTPUT = LOCK_ROOT / "ag36_test48_cpu_result_audit_v1.json"

EXPECTED = {
    TEST: "231ff1f0432be2bbc1eee9f879a5284f3d6c7633c5c674fdabd605495fd607f5",
    SPLIT_LOCK: "c4bac6f9eefdbe072b1ee9f6a44d7fd88df863139967c594fda1798a71c6fa8b",
    PROMPT_CATALOG: "e944ef5cbdda7c67930f92fc0ee6151fffbfc5d477d63c9886bed2a8a7fda066",
    SELECTION: "8aa3c5854e32aade4947fed14886a776823974646e0bb5c4cf3ce67d468c88ac",
    PRETEST_LOCK: "b0bb64a6068fe4151a5a606aafedc6f6b1413ae04f6c69b368bce462356d5e4a",
}
for path, expected_hash in EXPECTED.items():
    require(path.is_file(), f"Missing required artifact: {path}")
    require(sha256(path) == expected_hash, f"SHA-256 drift: {path}")
require(PROTOCOL.is_file(), "AG35 protocol is missing")
require(COMPARISON.is_file(), "AG35 is incomplete: comparison.json is missing; wait for AG35_COMPLETE")

test = json.loads(TEST.read_bytes())
split_lock = json.loads(SPLIT_LOCK.read_bytes())
selection = json.loads(SELECTION.read_bytes())
pretest_lock = json.loads(PRETEST_LOCK.read_bytes())
protocol = json.loads(PROTOCOL.read_bytes())
comparison = json.loads(COMPARISON.read_bytes())

require(len(test) == 48, "Expected exactly 48 locked test queries")
query_ids = [row["query_id"] for row in test]
require(len(set(query_ids)) == 48, "Duplicate query ID in test48")
test_by_id = {row["query_id"]: row for row in test}
video_ids = sorted({row["video_id"] for row in test})
require(len(video_ids) == 46, "Expected exactly 46 target videos")
require(split_lock.get("locked_evaluation_ready") is True, "AG30 split is not evaluation-ready")
require(split_lock.get("test", {}).get("sha256") == EXPECTED[TEST], "AG30 test binding drift")
require(pretest_lock.get("status") == "test48_final_pre_evaluation_configuration_locked",
        "AG34 pre-evaluation lock status drift")
require(pretest_lock.get("evaluation_executed") is False and
        pretest_lock.get("test_results_observed") is False,
        "AG34 was not a clean pre-evaluation lock")

strategies = ["target_centered", "highest_score_window", "minimum_unaffected_floor"]
require(protocol.get("scope") == "locked_test48_final_region_evaluation_once_only",
        "AG35 protocol scope drift")
require(list(protocol.get("strategies", [])) == strategies, "AG35 strategy order drift")
require(protocol.get("ag23_lock_sha256") == EXPECTED[PRETEST_LOCK], "AG35/AG34 lock mismatch")
require(protocol.get("ag33_comparison_sha256") == EXPECTED[SELECTION], "AG35/AG33 mismatch")
require(protocol.get("test_sha256") == EXPECTED[TEST], "AG35 test hash mismatch")
for key, expected_value in {
    "epsilon": 8 / 255,
    "steps": 15,
    "lambda1": 0.5,
    "lambda2": 0.0,
    "frame_budget": 16,
}.items():
    require(protocol.get(key) == expected_value, f"Locked parameter drift: {key}")
require(comparison.get("protocol") == protocol, "comparison.json protocol differs from protocol.json")

selection_rows = selection.get("results", [])
selection_by_id = {row["query_id"]: row for row in selection_rows}
require(len(selection_rows) == 48 and set(selection_by_id) == set(query_ids),
        "AG33 selection query set mismatch")

paired_rows = comparison.get("paired", [])
paired_by_id = {row["query_id"]: row for row in paired_rows}
require(len(paired_rows) == 48 and set(paired_by_id) == set(query_ids),
        "AG35 paired query set mismatch")
reports = comparison.get("results", [])
require(len(reports) == 3 and [report.get("strategy") for report in reports] == strategies,
        "AG35 must contain exactly the three locked strategies in order")

authority_hashes = {}
runtime_identities = {}
rows_by_strategy = {}
config_hashes = set()
for report in reports:
    strategy = report["strategy"]
    authority = RUN / strategy / "authority.jsonl"
    identity_path = RUN / strategy / "run_identity.json"
    require(authority.is_file() and identity_path.is_file(), f"Missing AG35 artifact: {strategy}")
    authority_hash = sha256(authority)
    require(authority_hash == report.get("authority_sha256"), f"Authority hash drift: {strategy}")
    identity = json.loads(identity_path.read_bytes())
    require(identity.get("protocol_sha256") == sha256(PROTOCOL), f"Protocol identity drift: {strategy}")
    require(identity.get("strategy") == strategy, f"Strategy identity drift: {strategy}")
    for key in ("torch", "cuda", "gpu"):
        require(report.get(key) == identity.get(key), f"Runtime identity mismatch: {strategy}/{key}")
    rows = [json.loads(line) for line in authority.read_text(encoding="utf-8").splitlines()
            if line.strip()]
    require(len(rows) == 48, f"{strategy}: expected 48 authority rows")
    require({row.get("query_id") for row in rows} == set(query_ids),
            f"{strategy}: authority query set mismatch")
    require(all(row.get("status") == "success" for row in rows),
            f"{strategy}: failure row present")
    require(all(row.get("condition") == "first_order" for row in rows),
            f"{strategy}: condition drift")
    rows_by_id = {row["query_id"]: row for row in rows}
    for query_id in query_ids:
        row = rows_by_id[query_id]
        expected_target = test_by_id[query_id]
        require(row.get("video_id") == expected_target.get("video_id"),
                f"{strategy}/{query_id}: target video mismatch")
        config_hashes.add(row.get("config_hash"))
        measurements = row.get("measurements", {})
        require(measurements.get("uses_mock") is False, f"{strategy}/{query_id}: mock result")
        require(measurements.get("device") == "cuda", f"{strategy}/{query_id}: non-CUDA result")
        require(measurements.get("changed_frame_count") == 16,
                f"{strategy}/{query_id}: frame budget drift")
        expected_positions = selection_by_id[query_id]["analysis"][strategy]["positions"]
        require(measurements.get("changed_positions_zero_based") == expected_positions,
                f"{strategy}/{query_id}: selected interval mismatch")
        require(measurements.get("perturbation_linf", float("inf")) <= 8 / 255 + 1e-5,
                f"{strategy}/{query_id}: epsilon violation")
        require(isinstance(row.get("rank"), int) and row["rank"] >= 1,
                f"{strategy}/{query_id}: invalid rank")
        for key in ("score", "clean_rank", "ssim", "flicker", "acceleration", "elapsed_time"):
            require(finite_number(measurements.get(key)), f"{strategy}/{query_id}: invalid {key}")
        require(measurements.get("psnr") is None or finite_number(measurements.get("psnr")),
                f"{strategy}/{query_id}: invalid psnr")
        paired = paired_by_id[query_id][strategy]
        require(row["rank"] == paired.get("rank"), f"{strategy}/{query_id}: paired rank mismatch")
        for key in ("score", "clean_rank", "psnr", "ssim", "flicker", "acceleration", "elapsed_time"):
            require(measurements.get(key) == paired.get(key),
                    f"{strategy}/{query_id}: paired measurement mismatch for {key}")
    require(report.get("rows") == rows, f"Embedded authority rows drift: {strategy}")
    authority_hashes[strategy] = authority_hash
    runtime_identities[strategy] = identity
    rows_by_strategy[strategy] = rows_by_id

require(len(config_hashes) == 1 and None not in config_hashes, "AG35 config hash is not uniform")
clean_ranks = np.array([paired_by_id[q][strategies[0]]["clean_rank"] for q in query_ids], dtype=int)
for strategy in strategies[1:]:
    require(np.array_equal(clean_ranks,
                           np.array([paired_by_id[q][strategy]["clean_rank"] for q in query_ids], dtype=int)),
            f"Clean-rank parity failed: {strategy}")

ranks = {
    strategy: np.array([paired_by_id[q][strategy]["rank"] for q in query_ids], dtype=float)
    for strategy in strategies
}
groups = [np.array([i for i, row in enumerate(test) if row["video_id"] == video_id])
          for video_id in video_ids]
rng = np.random.default_rng(20260908)
bootstrap_samples = [
    np.concatenate([groups[index] for index in rng.integers(len(groups), size=len(groups))])
    for _ in range(10000)
]

report = {
    "schema_version": 1,
    "status": "test48_post_evaluation_audit_passed",
    "artifact_class": "post_evaluation_cpu_integrity_statistics_and_paper_input",
    "source_hashes": {
        "test48": EXPECTED[TEST],
        "split_lock": EXPECTED[SPLIT_LOCK],
        "prompt_catalog": EXPECTED[PROMPT_CATALOG],
        "region_selection": EXPECTED[SELECTION],
        "pretest_lock": EXPECTED[PRETEST_LOCK],
        "ag35_protocol": sha256(PROTOCOL),
        "ag35_comparison": sha256(COMPARISON),
        "authorities": authority_hashes,
    },
    "query_count": 48,
    "video_count": 46,
    "strategy_count": 3,
    "successful_execution_count": 144,
    "failure_count": 0,
    "paired_denominator": 48,
    "config_hash": next(iter(config_hashes)),
    "runtime_identities": runtime_identities,
    "inference_executed": False,
    "test_split_modified": False,
    "bootstrap": {
        "method": "paired video-cluster bootstrap",
        "cluster_count": 46,
        "query_weighted": True,
        "draws": 10000,
        "seed": 20260908,
        "interval": "percentile 95%",
    },
    "metrics": {},
    "paired_comparisons": {},
}

for strategy in strategies:
    values = ranks[strategy]
    eligible10 = clean_ranks <= 10
    rank_shift = values - clean_ranks
    report["metrics"][strategy] = {
        "mean_rank": float(values.mean()),
        "median_rank": float(np.median(values)),
        "mrr": float(np.mean(1.0 / values)),
        "recall_at_1": float(np.mean(values <= 1)),
        "recall_at_5": float(np.mean(values <= 5)),
        "recall_at_10": float(np.mean(values <= 10)),
        "mean_rank_shift_from_clean": float(rank_shift.mean()),
        "median_rank_shift_from_clean": float(np.median(rank_shift)),
        "asr_at_10_numerator": int(np.sum((values > 10) & eligible10)),
        "asr_at_10_denominator": int(np.sum(eligible10)),
        "mean_psnr": float(np.mean([
            paired_by_id[q][strategy]["psnr"] for q in query_ids
            if paired_by_id[q][strategy]["psnr"] is not None
        ])),
        "mean_ssim": float(np.mean([paired_by_id[q][strategy]["ssim"] for q in query_ids])),
        "mean_elapsed_seconds": float(np.mean([
            paired_by_id[q][strategy]["elapsed_time"] for q in query_ids
        ])),
    }

selected_strategy = "minimum_unaffected_floor"
raw_p_values = {}
for baseline in strategies[:2]:
    delta = ranks[selected_strategy] - ranks[baseline]
    reciprocal_rank_delta = 1.0 / ranks[selected_strategy] - 1.0 / ranks[baseline]
    wins = int(np.sum(delta > 0))
    ties = int(np.sum(delta == 0))
    losses = int(np.sum(delta < 0))
    p_value = exact_two_sided_sign_test(wins, losses)
    raw_p_values[baseline] = p_value
    report["paired_comparisons"][baseline] = {
        "direction": "positive rank delta means stronger target-rank suppression by minimum_unaffected_floor",
        "wins_ties_losses": [wins, ties, losses],
        "non_tied_denominator": wins + losses,
        "mean_rank_delta": float(delta.mean()),
        "median_rank_delta": float(np.median(delta)),
        "rank_delta_ci95_video_cluster": np.quantile(
            [delta[index].mean() for index in bootstrap_samples], [0.025, 0.975]
        ).tolist(),
        "mean_mrr_delta": float(reciprocal_rank_delta.mean()),
        "mrr_delta_ci95_video_cluster": np.quantile(
            [reciprocal_rank_delta[index].mean() for index in bootstrap_samples], [0.025, 0.975]
        ).tolist(),
        "exact_two_sided_sign_test_p_unadjusted": p_value,
    }

for baseline, p_value in raw_p_values.items():
    adjusted = None if p_value is None else min(1.0, p_value * len(raw_p_values))
    report["paired_comparisons"][baseline].update({
        "bonferroni_family_size": 2,
        "bonferroni_adjusted_p": adjusted,
        "familywise_alpha": 0.05,
        "per_comparison_alpha": 0.025,
        "significant_familywise_0_05": adjusted is not None and adjusted <= 0.05,
    })

payload = canonical_bytes(report)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
if OUTPUT.exists():
    require(OUTPUT.read_bytes() == payload, "Existing immutable AG36 audit differs")
else:
    temporary = OUTPUT.with_suffix(OUTPUT.suffix + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, OUTPUT)

print("AG36_TEST48_RESULT_AUDIT_OK")
print(json.dumps({
    **report,
    "path": str(OUTPUT),
    "sha256": sha256(OUTPUT),
}, ensure_ascii=False, indent=2))


In [ ]:
%%stage AG37R
                                                                                          
                                                                  
                                                                             
                                                                          

import hashlib
import json
import os
import re
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

ROOT = Path("/content/reproduction_inputs/artifacts")
AG30_ROOT = ROOT / "evaluation/round1_evaluation"
AGB4_ROOT = ROOT / "development/extension_query_promotion"
OUTPUT = ROOT / "evaluation/round2_evaluation"
V1_MANIFEST = (
    ROOT
    / "evaluation/round2_lock_superseded"
    / "test48_batch_a_extension_lock_manifest.json"
)

DEV10 = AG30_ROOT / "dev10.json"
TEST48 = AG30_ROOT / "test48.json"
AG30_MANIFEST = AG30_ROOT / "dev10_test48_split_lock_manifest.json"
AGB4_ROWS = AGB4_ROOT / "batch_a_promoted_annotations.jsonl"
AGB4_MANIFEST = AGB4_ROOT / "batch_a_promoted_manifest.json"

BATCH_EXTENSION = OUTPUT / "batch_a_extension.json"
COMBINED_TEST = OUTPUT / "test_extended.json"
MANIFEST = OUTPUT / "test48_batch_a_extension_lock_manifest.json"

EXPECTED_AG30_MANIFEST_SHA256 = "c4bac6f9eefdbe072b1ee9f6a44d7fd88df863139967c594fda1798a71c6fa8b"
EXPECTED_AGB4_ROWS_SHA256 = "8c4d90d55ab54c64bdb65a833783cdc152d334ff2ae62d81af0814e40e3ef6c3"
EXPECTED_AGB4_MANIFEST_SHA256 = "f5c6cc78c3530c974df81acda67dffb42fe714d84fc377ee3bb62208aac46dec"
EXPECTED_SUPERSEDED_V1_MANIFEST_SHA256 = "82beef28863a2ed5c418f7d0938e739f39cc575f566851237c3cd04599816c87"


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))


def load_jsonl(path):
    return [
        json.loads(line)
        for line in Path(path).read_text(encoding="utf-8-sig").splitlines()
        if line.strip()
    ]


def pretty(value):
    return (json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + "\n").encode("utf-8")


def canonical(value):
    return (
        json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n"
    ).encode("utf-8")


def immutable_write(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        require(path.read_bytes() == payload, f"Immutable artifact drift: {path}")
        return
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, path)


for required_path in (DEV10, TEST48, AG30_MANIFEST, AGB4_ROWS, AGB4_MANIFEST):
    require(required_path.is_file(), f"Missing required artifact: {required_path}")

require(sha256(AG30_MANIFEST) == EXPECTED_AG30_MANIFEST_SHA256, "AG30 manifest hash drift")
require(sha256(AGB4_ROWS) == EXPECTED_AGB4_ROWS_SHA256, "AGB4 promoted rows hash drift")
require(sha256(AGB4_MANIFEST) == EXPECTED_AGB4_MANIFEST_SHA256, "AGB4 manifest hash drift")

dev10 = load_json(DEV10)
test48 = load_json(TEST48)
ag30 = load_json(AG30_MANIFEST)
batch_rows = load_jsonl(AGB4_ROWS)
agb4 = load_json(AGB4_MANIFEST)

require(isinstance(dev10, list) and len(dev10) == 10, "Expected unchanged dev10")
require(isinstance(test48, list) and len(test48) == 48, "Expected unchanged test48")
require(ag30.get("status") == "organizer58_dev10_test48_locked_before_evaluation", "Bad AG30 status")
require(ag30.get("locked_evaluation_ready") is True, "AG30 test48 is not locked")
require(ag30.get("dev", {}).get("sha256") == sha256(DEV10), "AG30 dev10 binding mismatch")
require(ag30.get("test", {}).get("sha256") == sha256(TEST48), "AG30 test48 binding mismatch")
require(agb4.get("promotion_ready") is True, "AGB4 is not promotion-ready")
require(agb4.get("locked_evaluation_ready") is False, "Unexpected AGB4 final-lock claim")
require(
    agb4.get("retrieval_derived_ground_truth") is False,
    "AGB4 does not prove retrieval-independent ground truth",
)
require(agb4.get("official_mapping_verified") is True, "AGB4 mapping is not verified")
require(
    agb4.get("canonical_annotation_validation", {}).get("status") == "valid",
    "AGB4 canonical annotation validation is not valid",
)
require(agb4.get("promoted_count") == 96 and len(batch_rows) == 96, "Expected 96 AGB4 records")
require(agb4.get("excluded_count") == 4, "Expected four ambiguous exclusions")
require(agb4.get("promoted_annotations", {}).get("sha256") == sha256(AGB4_ROWS), "AGB4 payload binding mismatch")

dev_ids = {row["query_id"] for row in dev10}
test48_ids = {row["query_id"] for row in test48}
dev_videos = {row["video_id"] for row in dev10}
require(len(dev_ids) == 10 and len(test48_ids) == 48, "Duplicate organizer query IDs")
require(not (dev_ids & test48_ids), "Organizer dev/test query leakage")
require(not (dev_videos & {row["video_id"] for row in test48}), "Organizer dev/test video leakage")

batch_source_ids = [row.get("query_id") for row in batch_rows]
batch_candidate_ids = [row.get("candidate_id") for row in batch_rows]
require(len(set(batch_source_ids)) == len(batch_rows), "Duplicate Batch A source query IDs")
require(len(set(batch_candidate_ids)) == len(batch_rows), "Duplicate Batch A candidate IDs")
require(all(re.fullmatch(r"B\d{3}", value or "") for value in batch_candidate_ids), "Bad Batch A candidate ID")
require(all(row.get("unique_match_pass") is True for row in batch_rows), "Batch A contains a non-pass record")
require(
    all(
        hashlib.sha256(row.get("query_text_vi", "").encode("utf-8")).hexdigest()
        == row.get("query_text_vi_sha256")
        and hashlib.sha256(row.get("query_text_en", "").encode("utf-8")).hexdigest()
        == row.get("query_text_en_sha256")
        and row.get("query_text_sha256") == row.get("query_text_en_sha256")
        for row in batch_rows
    ),
    "Batch A query text binding mismatch",
)
require(
    all(
        row.get("ground_truth_status") == "promotion_ready_provenance_and_uniqueness_resolved"
        and row.get("promotion_status") == "promotion_ready_not_final_locked_evaluation"
        for row in batch_rows
    ),
    "Batch A contains a record outside the AGB4 promotion-ready state",
)

extension = []
excluded_dev_video = []
for source in sorted(batch_rows, key=lambda row: row["candidate_id"]):
    if source["video_id"] in dev_videos:
        excluded_dev_video.append({
            "source_query_id": source["query_id"],
            "candidate_id": source["candidate_id"],
            "video_id": source["video_id"],
            "reason": "target_video_overlaps_locked_dev10",
        })
        continue

    candidate_number = int(source["candidate_id"][1:])
    evaluation_query_id = f"HCMC-B1-Q{1000 + candidate_number:04d}"
    require(evaluation_query_id not in dev_ids | test48_ids, "Namespaced query ID collision")
    original_hash = hashlib.sha256(canonical(source)).hexdigest()
    row = dict(source)
    row.update({
        "query_id": evaluation_query_id,
                                                                          
                                                                              
        "query_text_sha256": source["query_text_vi_sha256"],
        "retrieval_query_text_sha256": source["query_text_en_sha256"],
        "agb4_query_text_sha256": source["query_text_sha256"],
        "batch_a_source_query_id": source["query_id"],
        "batch_a_source_candidate_id": source["candidate_id"],
        "batch_a_source_record_sha256": original_hash,
        "evaluation_cohort": "batch_a_extension",
        "evaluation_lock_basis": "agb4_promotion_plus_dev_video_disjoint_audit",
        "human_hard_negative_review_performed": False,
        "delayed_self_audit_performed": False,
        "locked_evaluation_ready": True,
    })
    extension.append(row)

extension_ids = {row["query_id"] for row in extension}
extension_videos = {row["video_id"] for row in extension}
require(len(extension_ids) == len(extension), "Duplicate extension query IDs")
require(not (extension_ids & (dev_ids | test48_ids)), "Extension query ID leakage")
require(not (extension_videos & dev_videos), "Extension target video overlaps dev10")
require(len(extension) > 0, "No Batch A record remains after dev-video exclusion")

combined = [dict(row, evaluation_cohort="organizer_test48") for row in test48] + extension
combined = sorted(combined, key=lambda row: row["query_id"])
require(len({row["query_id"] for row in combined}) == len(combined), "Combined query ID collision")
require(not ({row["video_id"] for row in combined} & dev_videos), "Combined test overlaps dev10")

if V1_MANIFEST.is_file():
    require(
        sha256(V1_MANIFEST) == EXPECTED_SUPERSEDED_V1_MANIFEST_SHA256,
        "Superseded AG37 v1 manifest hash drift",
    )

immutable_write(BATCH_EXTENSION, pretty(extension))
immutable_write(COMBINED_TEST, pretty(combined))

manifest = {
    "schema_version": 1,
    "status": "test48_batch_a_extension_v2_locked_before_batch_a_attack",
    "artifact_class": "preserved_historical_test48_plus_pre_attack_batch_a_extension",
    "supersedes": {
        "artifact": str(V1_MANIFEST),
        "sha256": EXPECTED_SUPERSEDED_V1_MANIFEST_SHA256,
        "reason": "v1 retained AGB4 English query_text_sha256, which is incompatible with the shared prompt-centroid runner's canonical Vietnamese binding",
    },
    "source_hashes": {
        "ag30_manifest": sha256(AG30_MANIFEST),
        "dev10": sha256(DEV10),
        "test48": sha256(TEST48),
        "agb4_manifest": sha256(AGB4_MANIFEST),
        "agb4_promoted_annotations": sha256(AGB4_ROWS),
    },
    "selected_condition": ag30.get("selected_condition"),
    "selected_condition_config_hash": ag30.get("selected_condition_config_hash"),
    "selected_attack_parameters": ag30.get("selected_attack_parameters"),
    "retrieval_parameters": ag30.get("retrieval_parameters"),
    "dev10": {"sha256": sha256(DEV10), "query_count": 10, "video_count": len(dev_videos)},
    "historical_test48": {
        "sha256": sha256(TEST48),
        "query_count": 48,
        "preserved_unchanged": True,
        "results_already_observed": True,
    },
    "batch_a_extension": {
        "path": str(BATCH_EXTENSION),
        "sha256": sha256(BATCH_EXTENSION),
        "source_promoted_count": 96,
        "included_count": len(extension),
        "excluded_for_dev_video_overlap": excluded_dev_video,
        "query_id_namespace": "Bxxx maps to HCMC-B1-Q1xxx",
        "attack_results_observed_before_lock": False,
        "human_hard_negative_review_performed": False,
        "delayed_self_audit_performed": False,
        "uniqueness_basis": "AGB2R machine audit plus AGB3R human review of flagged exceptions",
    },
    "combined_test": {
        "path": str(COMBINED_TEST),
        "sha256": sha256(COMBINED_TEST),
        "query_count": len(combined),
        "video_count": len({row["video_id"] for row in combined}),
    },
    "query_disjoint_from_dev": True,
    "video_disjoint_from_dev": True,
    "uses_mock": False,
    "locked_evaluation_ready": True,
    "runner_query_binding_compatible": True,
    "evaluation_executed": False,
    "reporting_requirement": "Report organizer test48, Batch A extension, and pooled results separately.",
    "scope_boundary": "The pooled cohort was not wholly locked before historical test48 results; only the Batch A extension was locked before its attacks.",
}
immutable_write(MANIFEST, canonical(manifest))

print("AG37R_BATCH_A_TEST48_EXTENSION_LOCK_V2_OK")
print(json.dumps({
    "historical_test48_count": 48,
    "batch_a_source_promoted_count": 96,
    "batch_a_extension_count": len(extension),
    "excluded_for_dev_video_overlap_count": len(excluded_dev_video),
    "combined_test_count": len(combined),
    "combined_test_sha256": sha256(COMBINED_TEST),
    "manifest_path": str(MANIFEST),
    "manifest_sha256": sha256(MANIFEST),
    "locked_evaluation_ready": True,
    "runner_query_binding_compatible": True,
    "uses_mock": False,
}, ensure_ascii=False, indent=2))


In [ ]:
%%stage AG38
                                                                                   
                                                                  
                                                                             
                                                                          

import hashlib
import importlib.metadata
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

ROOT = Path("/content/reproduction_inputs/artifacts")
LOCK_ROOT = ROOT / "evaluation/round2_evaluation"
EXTENSION = LOCK_ROOT / "batch_a_extension.json"
COMBINED = LOCK_ROOT / "test_extended.json"
EXTENSION_LOCK = LOCK_ROOT / "test48_batch_a_extension_lock_manifest.json"
TOKENIZER_CACHE = ROOT / "artifacts/models/ViT-B-16-SigLIP_webli/tokenizer_cache"
OUTPUT = LOCK_ROOT / "prompt_centroid_v1"
PROMPT_CATALOG = OUTPUT / "batch_a_extension_prompt_catalog.jsonl"
MANIFEST = OUTPUT / "batch_a_extension_prompt_catalog_manifest.json"

EXPECTED_EXTENSION_LOCK_SHA256 = "46fccfad9678aaae2f4ee035c59fe1e60f6c49ec2b361dacc6b1e7a47fde14e9"
EXPECTED_COMBINED_SHA256 = "cec5408c960e9d78738dd39506c87942deb3e9b6ca0321692c5f283841c5063d"
EXPECTED_EXTENSION_COUNT = 93
PROMPT_CONTEXT_LIMIT = 64
OPEN_CLIP_VERSION = "3.3.0"
TOKENIZER_SNAPSHOT_REVISION = "41f575766f40e752fdd1383e9565b7f02388c1c4"


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def text_sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def canonical(value):
    return (
        json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
        + "\n"
    ).encode("utf-8")


def immutable_write(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        require(path.read_bytes() == payload, f"Immutable artifact drift: {path}")
        return
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, path)


def asset_manifest(root):
    root = Path(root)
    files = [
        path
        for path in root.rglob("*")
        if path.is_file()
        and ".cache" not in path.relative_to(root).parts
        and "__pycache__" not in path.relative_to(root).parts
    ]
    require(files, f"No tokenizer assets found under {root}")
    entries = [
        {
            "path": path.relative_to(root).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in sorted(files)
    ]
    return entries, hashlib.sha256(canonical(entries)).hexdigest()


for required_path in (EXTENSION, COMBINED, EXTENSION_LOCK, TOKENIZER_CACHE):
    require(required_path.exists(), f"Missing required artifact: {required_path}")
require(
    sha256(EXTENSION_LOCK) == EXPECTED_EXTENSION_LOCK_SHA256,
    "AG37R v2 manifest hash drift",
)
require(sha256(COMBINED) == EXPECTED_COMBINED_SHA256, "Combined test hash drift")

lock = json.loads(EXTENSION_LOCK.read_text(encoding="utf-8-sig"))
extension = json.loads(EXTENSION.read_text(encoding="utf-8-sig"))
combined = json.loads(COMBINED.read_text(encoding="utf-8-sig"))
require(
    lock.get("status") == "test48_batch_a_extension_v2_locked_before_batch_a_attack",
    "AG37R v2 is not locked",
)
require(lock.get("locked_evaluation_ready") is True, "AG37R v2 is not evaluation-ready")
require(lock.get("runner_query_binding_compatible") is True, "AG37R runner binding missing")
require(lock.get("evaluation_executed") is False, "Batch A evaluation was already marked executed")
require(lock.get("uses_mock") is False, "AG37R v2 uses mock evidence")
require(
    lock.get("batch_a_extension", {}).get("sha256") == sha256(EXTENSION),
    "AG37R does not bind extension bytes",
)
require(
    lock.get("combined_test", {}).get("sha256") == sha256(COMBINED),
    "AG37R does not bind combined bytes",
)
require(
    isinstance(extension, list)
    and len(extension) == EXPECTED_EXTENSION_COUNT
    and len({row.get("query_id") for row in extension}) == EXPECTED_EXTENSION_COUNT,
    "Expected 93 unique Batch A extension queries",
)
require(isinstance(combined, list) and len(combined) == 141, "Expected combined test-141")

for row in extension:
    query_id = row.get("query_id")
    vi = row.get("query_text_vi")
    en = row.get("query_text_en")
    require(isinstance(query_id, str) and query_id, "Missing extension query ID")
    require(isinstance(vi, str) and vi.strip(), f"{query_id}: missing Vietnamese wording")
    require(isinstance(en, str) and en.strip(), f"{query_id}: missing English wording")
    require(text_sha256(vi) == row.get("query_text_vi_sha256"), f"{query_id}: VI hash drift")
    require(text_sha256(en) == row.get("query_text_en_sha256"), f"{query_id}: EN hash drift")
    require(row.get("query_text_sha256") == row.get("query_text_vi_sha256"),
            f"{query_id}: runner canonical binding drift")
    require(row.get("retrieval_query_text_sha256") == row.get("query_text_en_sha256"),
            f"{query_id}: retrieval text binding drift")
    require(row.get("locked_evaluation_ready") is True, f"{query_id}: record is not locked")

missing_packages = []
for module_name, package_name in (
    ("transformers", "transformers"),
    ("open_clip", f"open_clip_torch=={OPEN_CLIP_VERSION}"),
):
    if importlib.util.find_spec(module_name) is None:
        missing_packages.append(package_name)
if missing_packages:
    print("AG38_INSTALLING", missing_packages, flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
if importlib.metadata.version("open_clip_torch") != OPEN_CLIP_VERSION:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", f"open_clip_torch=={OPEN_CLIP_VERSION}"]
    )

import open_clip

snapshot = (
    TOKENIZER_CACHE
    / "models--timm--ViT-B-16-SigLIP"
    / "snapshots"
    / TOKENIZER_SNAPSHOT_REVISION
)
required_snapshot_files = (
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer_config.json",
)
require(
    all((snapshot / name).is_file() for name in required_snapshot_files),
    f"Incomplete pinned SigLIP tokenizer snapshot: {snapshot}",
)
model_config = open_clip.get_model_config("ViT-B-16-SigLIP")
tokenizer_kwargs = model_config["text_cfg"].get("tokenizer_kwargs", {})
wrapper = open_clip.tokenizer.HFTokenizer(
    str(snapshot),
    context_length=PROMPT_CONTEXT_LIMIT,
    clean=tokenizer_kwargs.get("clean", "whitespace"),
    local_files_only=True,
)
tokenizer = wrapper.tokenizer
tokenizer_assets, tokenizer_assets_sha256 = asset_manifest(snapshot)


def token_count(text):
    encoded = tokenizer(
        text,
        add_special_tokens=True,
        padding=False,
        truncation=False,
        return_attention_mask=False,
    )
    return len(encoded["input_ids"])


def exact_chunks(text):
    normalized = " ".join(text.split())
    require(normalized == text.strip(), "Locked English wording contains unstable whitespace")
    if token_count(normalized) <= PROMPT_CONTEXT_LIMIT:
        return [normalized]
    chunks = []
    current = ""
    for word in normalized.split(" "):
        candidate = " ".join(filter(None, (current, word)))
        if token_count(candidate) <= PROMPT_CONTEXT_LIMIT:
            current = candidate
        else:
            require(current, f"Single English word exceeds tokenizer context: {word!r}")
            chunks.append(current)
            current = word
    if current:
        chunks.append(current)
    require(" ".join(chunks) == normalized, "English chunking changed locked wording")
    return chunks


catalog = []
chunked_query_ids = []
for row in sorted(extension, key=lambda item: item["query_id"]):
    query_id = row["query_id"]
    chunks = exact_chunks(row["query_text_en"])
    if len(chunks) > 1:
        chunked_query_ids.append(query_id)
    prompts = []
    for index, text in enumerate(chunks, start=1):
        count = token_count(text)
        require(1 <= count <= PROMPT_CONTEXT_LIMIT, f"{query_id}: prompt exceeds context")
        prompts.append({
            "prompt_role": "locked_authored_english" if len(chunks) == 1
                           else f"locked_authored_english_chunk_{index:02d}",
            "text_en": text,
            "text_en_sha256": text_sha256(text),
            "siglip_token_count": count,
        })
    catalog.append({
        "schema_version": 1,
        "query_id": query_id,
        "canonical_query_text_vi_sha256": row["query_text_sha256"],
        "locked_authored_english_sha256": row["query_text_en_sha256"],
        "prompt_derivation": "exact_locked_human_authored_english_no_translation_or_paraphrase",
        "retrieval_prompts": prompts,
    })

require(len(catalog) == EXPECTED_EXTENSION_COUNT, "Prompt catalog count mismatch")
catalog_payload = b"".join(canonical(row) for row in catalog)
immutable_write(PROMPT_CATALOG, catalog_payload)

manifest = {
    "schema_version": 1,
    "status": "batch_a_extension_prompt_catalog_locked_before_evaluation",
    "artifact_class": "derived_english_retrieval_prompts_not_ground_truth",
    "extension_lock_sha256": sha256(EXTENSION_LOCK),
    "extension_sha256": sha256(EXTENSION),
    "combined_test_sha256": sha256(COMBINED),
    "query_count": len(catalog),
    "prompt_count": sum(len(row["retrieval_prompts"]) for row in catalog),
    "chunked_query_count": len(chunked_query_ids),
    "chunked_query_ids": chunked_query_ids,
    "prompt_context_limit": PROMPT_CONTEXT_LIMIT,
    "prompt_derivation": "exact_locked_human_authored_english_no_translation_or_paraphrase",
    "canonical_query_text_modified": False,
    "english_query_text_modified": False,
    "tokenizer": {
        "model": "ViT-B-16-SigLIP",
        "snapshot_revision": TOKENIZER_SNAPSHOT_REVISION,
        "assets": tokenizer_assets,
        "assets_sha256": tokenizer_assets_sha256,
    },
    "prompt_catalog_path": str(PROMPT_CATALOG),
    "prompt_catalog_sha256": sha256(PROMPT_CATALOG),
    "ready_for_batch_a_extension_evaluation": True,
    "evaluation_executed": False,
    "batch_a_results_observed": False,
    "uses_mock": False,
}
immutable_write(MANIFEST, canonical(manifest))

print("AG38_BATCH_A_EXTENSION_PROMPT_CATALOG_LOCK_OK")
print(json.dumps({
    "extension_count": len(extension),
    "prompt_count": manifest["prompt_count"],
    "chunked_query_count": manifest["chunked_query_count"],
    "prompt_catalog": str(PROMPT_CATALOG),
    "prompt_catalog_sha256": sha256(PROMPT_CATALOG),
    "manifest": str(MANIFEST),
    "manifest_sha256": sha256(MANIFEST),
    "ready_for_batch_a_extension_evaluation": True,
    "batch_a_results_observed": False,
    "uses_mock": False,
}, ensure_ascii=False, indent=2))


In [ ]:
%%stage AG39
                       
                                                                           
                                                                      
                                                                               
                                                               

import hashlib
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import torch
from google.colab import drive, userdata

drive.mount("/content/drive")
os.environ.update(
    HF_HUB_OFFLINE="1",
    TRANSFORMERS_OFFLINE="1",
    HF_HUB_DISABLE_TELEMETRY="1",
    HF_HUB_DISABLE_IMPLICIT_TOKEN="1",
)

SOURCE_COMMIT = "ab526c50f491bae440ea697fca72adcb10c687f3"
ROOT = Path("/content/reproduction_inputs/artifacts")
DATA_ROOT = Path("/content/reproduction_inputs/dataset")
LOCK_ROOT = ROOT / "evaluation/round2_evaluation"
EXTENSION = LOCK_ROOT / "batch_a_extension.json"
EXTENSION_LOCK = LOCK_ROOT / "test48_batch_a_extension_lock_manifest.json"
PROMPT_CATALOG = LOCK_ROOT / "prompt_centroid_v1/batch_a_extension_prompt_catalog.jsonl"
PROMPT_MANIFEST = LOCK_ROOT / "prompt_centroid_v1/batch_a_extension_prompt_catalog_manifest.json"
OUTPUT = LOCK_ROOT / "ag39_batch_a_region_selection_v1"

CACHE_ROOT = ROOT / "frame_cache/model_frame_cache"
CACHE_MANIFEST = CACHE_ROOT / "manifest.json"
EXPECTED_VIDEO_IDS = ROOT / "catalog/expected_video_ids.json"
DATASET_MANIFEST = (
    ROOT
    / "frame_cache/repaired_frames/L22_V011"
    / "target_dataset_manifest_L22_V011_046_repair.json"
)
CHECKPOINT = ROOT / "artifacts/models/ViT-B-16-SigLIP_webli/open_clip_model.safetensors"
TOKENIZER_CACHE = ROOT / "artifacts/models/ViT-B-16-SigLIP_webli/tokenizer_cache"

EXPECTED = {
    EXTENSION_LOCK: "46fccfad9678aaae2f4ee035c59fe1e60f6c49ec2b361dacc6b1e7a47fde14e9",
    PROMPT_CATALOG: "499cc0fb6f80223799a18999735c46e6331479147af64bca7815d80aaa0ad148",
    PROMPT_MANIFEST: "2aec23a3c9214e7dc60f3bf331bed4eb2d2869c01104cc2e6365205fab6543a0",
    EXPECTED_VIDEO_IDS: "b47517d27e92f5b18d65579964c2cf8ea31454e3853b31aa939f3d2bb9948feb",
    CACHE_MANIFEST: "5cc4b25610bd433a2abcc38a81dc234dc09bc1f83ecfc4a8dfe8b248e9c7b2a4",
    DATASET_MANIFEST: "9d3a579384bd8a94ab405a1bfbb39c065705140b593da310176f3f9c7d9351ac",
    CHECKPOINT: "81942ea8c09b9e41963357cca5d2682118bf7eb7491c2ae51a28bc4f4f95b194",
}
EXPECTED_MODEL_SHA256 = "1d62e634a2ee5092794b3b3fbfeb07b0d9bdc3727b22d19a96ea2e25e654443a"
EXPECTED_QUERY_COUNT = 93
WINDOW_SIZE = 16
STRIDE = 1


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical(value):
    return (
        json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
        + "\n"
    ).encode("utf-8")


def immutable_write(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        require(path.read_bytes() == payload, f"Immutable artifact drift: {path}")
        return
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, path)


for path, expected_hash in EXPECTED.items():
    require(path.is_file(), f"Missing required artifact: {path}")
    require(sha256(path) == expected_hash, f"SHA-256 drift: {path}")
for path in (DATA_ROOT, CACHE_ROOT, TOKENIZER_CACHE):
    require(path.exists(), f"Missing required directory: {path}")

lock = json.loads(EXTENSION_LOCK.read_text(encoding="utf-8-sig"))
prompt_manifest = json.loads(PROMPT_MANIFEST.read_text(encoding="utf-8-sig"))
rows = json.loads(EXTENSION.read_text(encoding="utf-8-sig"))
extension_hash = sha256(EXTENSION)
require(
    lock.get("status") == "test48_batch_a_extension_v2_locked_before_batch_a_attack"
    and lock.get("locked_evaluation_ready") is True
    and lock.get("evaluation_executed") is False,
    "AG37R v2 lock is not ready",
)
require(lock.get("batch_a_extension", {}).get("sha256") == extension_hash,
        "AG37R extension binding drift")
require(
    prompt_manifest.get("status") == "batch_a_extension_prompt_catalog_locked_before_evaluation"
    and prompt_manifest.get("ready_for_batch_a_extension_evaluation") is True
    and prompt_manifest.get("batch_a_results_observed") is False,
    "AG38 prompt lock is not ready",
)
require(prompt_manifest.get("extension_sha256") == extension_hash,
        "AG38 extension binding drift")
require(prompt_manifest.get("prompt_catalog_sha256") == sha256(PROMPT_CATALOG),
        "AG38 prompt payload binding drift")
require(
    isinstance(rows, list)
    and len(rows) == EXPECTED_QUERY_COUNT
    and len({row.get("query_id") for row in rows}) == EXPECTED_QUERY_COUNT,
    "Expected 93 unique extension queries",
)
require(all(row.get("locked_evaluation_ready") is True for row in rows),
        "Extension contains an unlocked query")

try:
    open_clip_version = importlib.metadata.version("open_clip_torch")
except importlib.metadata.PackageNotFoundError:
    open_clip_version = None
if open_clip_version != "3.3.0":
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "open_clip_torch==3.3.0"]
    )

token = userdata.get("GITHUB_TOKEN")
require(isinstance(token, str) and token.strip(), "Add GITHUB_TOKEN to Colab Secrets")
runtime_parent = Path("/content/stv_ag39_runtime")
runtime_zip = Path("/content/stv_ag39_runtime.zip")
if runtime_parent.exists():
    shutil.rmtree(runtime_parent)
runtime_parent.mkdir(parents=True)
request = urllib.request.Request(
    f"https://api.github.com/repos/haruxne/STV-Attack/zipball/{SOURCE_COMMIT}",
    headers={
        "Authorization": f"Bearer {token.strip()}",
        "Accept": "application/vnd.github+json",
    },
)
with urllib.request.urlopen(request, timeout=120) as response:
    runtime_zip.write_bytes(response.read())
with zipfile.ZipFile(runtime_zip) as archive:
    archive.extractall(runtime_parent)
runtime_roots = [path for path in runtime_parent.iterdir() if path.is_dir()]
require(
    len(runtime_roots) == 1 and (runtime_roots[0] / "src").is_dir(),
    "Unexpected repository archive",
)
runtime_root = runtime_roots[0]
sys.path.insert(0, str(runtime_root))
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

from src.clip_windows import build_sliding_windows, target_centered_window
from src.embedding_cache import _load_and_validate_matrix
from src.experiment_runner import (
    OpenClipSiglipAdapter,
    canonical_model_identity,
    encode_query,
    load_embedding_cache,
    load_expected_video_ids,
    load_query_prompts,
    model_identity_sha256,
)
from src.retrieval import VideoFrameEmbeddings, _l2_normalize_rows, prepare_video

config = {
    "window_size": WINDOW_SIZE,
    "stride": STRIDE,
    "query_text_field": "query_text_en",
    "query_representation": {
        "mode": "english_prompt_centroid",
        "catalog_path": str(PROMPT_CATALOG),
        "catalog_sha256": EXPECTED[PROMPT_CATALOG],
    },
}
prompts = load_query_prompts(config, rows)
expected_ids, _ = load_expected_video_ids(
    EXPECTED_VIDEO_IDS,
    expected_sha256=EXPECTED[EXPECTED_VIDEO_IDS],
)
print("AG39_LOADING_CACHE", flush=True)
cache = load_embedding_cache(
    CACHE_MANIFEST,
    cache_root=CACHE_ROOT,
    expected_manifest_sha256=EXPECTED[CACHE_MANIFEST],
    expected_dataset_hash=EXPECTED[DATASET_MANIFEST],
    expected_model_hash=EXPECTED_MODEL_SHA256,
    expected_video_ids=expected_ids,
)
adapter = OpenClipSiglipAdapter.load_local(
    checkpoint_path=CHECKPOINT,
    tokenizer_cache_dir=TOKENIZER_CACHE,
    device="cpu",
    expected_checkpoint_sha256=EXPECTED[CHECKPOINT],
)
require(
    model_identity_sha256(canonical_model_identity(adapter)) == EXPECTED_MODEL_SHA256,
    "Model identity drift",
)

protocol = {
    "schema_version": 1,
    "scope": "locked_batch_a_extension_cache_geometry_selection_not_attack_success",
    "source_commit": SOURCE_COMMIT,
    "selection_algorithm": "exhaustive_contiguous_interval_v1",
    "strategies": [
        "target_centered",
        "highest_score_window",
        "minimum_unaffected_floor",
    ],
    "budget": WINDOW_SIZE,
    "contiguous_only": True,
    "window_size": WINDOW_SIZE,
    "stride": STRIDE,
    "extension_lock_sha256": EXPECTED[EXTENSION_LOCK],
    "extension_sha256": extension_hash,
    "cache_sha256": EXPECTED[CACHE_MANIFEST],
    "prompt_sha256": EXPECTED[PROMPT_CATALOG],
    "prompt_manifest_sha256": EXPECTED[PROMPT_MANIFEST],
    "dataset_manifest_sha256": EXPECTED[DATASET_MANIFEST],
    "model_sha256": EXPECTED_MODEL_SHA256,
    "evaluation_executed": False,
    "batch_a_results_observed": False,
    "uses_mock": False,
}
PROTOCOL_PATH = OUTPUT / "protocol.json"
immutable_write(PROTOCOL_PATH, canonical(protocol))
protocol_hash = sha256(PROTOCOL_PATH)
cache_by_video = {record.video_id: record for record in cache.records}


def analyze_regions(scores, windows, frame_count, target_positions):
    def evaluate(start):
        unaffected = [
            index
            for index, window in enumerate(windows)
            if window[-1] < start or window[0] >= start + WINDOW_SIZE
        ]
        return {
            "start": start,
            "positions": list(range(start, start + WINDOW_SIZE)),
            "unaffected_score_floor": max(
                (float(scores[index]) for index in unaffected),
                default=None,
            ),
            "unaffected_count": len(unaffected),
        }

    choices = [evaluate(start) for start in range(frame_count - WINDOW_SIZE + 1)]
    best = min(
        choices,
        key=lambda item: (
            float("-inf")
            if item["unaffected_score_floor"] is None
            else item["unaffected_score_floor"],
            item["start"],
        ),
    )
    top_window_index = max(range(len(scores)), key=lambda index: float(scores[index]))
    target_start = min(target_positions)
    require(
        list(target_positions) == list(range(target_start, target_start + WINDOW_SIZE)),
        "Target-centered interval is not contiguous",
    )
    return {
        "scope": "contiguous_intervals_only_same_16_frame_budget",
        "target_centered": evaluate(target_start),
        "highest_score_window": evaluate(
            min(windows[top_window_index][0], frame_count - WINDOW_SIZE)
        ),
        "minimum_unaffected_floor": best,
        "intervals_evaluated": len(choices),
    }


reports = []
for index, row in enumerate(sorted(rows, key=lambda item: item["query_id"]), start=1):
    query_id = row["query_id"]
    destination = OUTPUT / f"{query_id}.json"
    if destination.is_file():
        report = json.loads(destination.read_text(encoding="utf-8-sig"))
        require(report.get("protocol_sha256") == protocol_hash, f"{query_id}: resume drift")
        reports.append(report)
        print("AG39_RESUMED", index, "/", EXPECTED_QUERY_COUNT, query_id, flush=True)
        continue

    print("AG39_QUERY", index, "/", EXPECTED_QUERY_COUNT, query_id, flush=True)
    video_id = row["video_id"]
    require(video_id in cache_by_video, f"{query_id}: video absent from cache")
    matrix = _load_and_validate_matrix(cache.root, cache_by_video[video_id])
    frame_embeddings = torch.from_numpy(np.array(matrix, copy=True))
    windows = build_sliding_windows(
        len(frame_embeddings),
        window_size=WINDOW_SIZE,
        stride=STRIDE,
    )
    prepared = prepare_video(
        VideoFrameEmbeddings(
            video_id=video_id,
            clips=tuple(frame_embeddings[list(window)] for window in windows),
        )
    )
    with torch.inference_mode():
        query_embedding = encode_query(adapter, prompts[query_id], centroid=True).cpu()
        clip_embeddings = _l2_normalize_rows(prepared.embeddings, name="cached clips")
        query_embedding = _l2_normalize_rows(query_embedding, name="query").to(clip_embeddings)
        scores = torch.mv(clip_embeddings, query_embedding).tolist()
    target_positions = sorted(
        set(
            target_centered_window(
                len(frame_embeddings),
                row["target_keyframe_n"] - 1,
                window_size=WINDOW_SIZE,
            )
        )
    )
    report = {
        "query_id": query_id,
        "video_id": video_id,
        "clean_score": max(scores),
        "protocol_sha256": protocol_hash,
        "analysis": analyze_regions(
            scores,
            windows,
            len(frame_embeddings),
            target_positions,
        ),
    }
    immutable_write(destination, canonical(report))
    reports.append(report)

require(
    len(reports) == EXPECTED_QUERY_COUNT
    and len({report["query_id"] for report in reports}) == EXPECTED_QUERY_COUNT,
    "Incomplete Batch A region selection",
)
COMPARISON = OUTPUT / "comparison.json"
immutable_write(
    COMPARISON,
    canonical({
        "protocol": protocol,
        "results": sorted(reports, key=lambda item: item["query_id"]),
    }),
)

print("AG39_BATCH_A_EXTENSION_REGION_SELECTION_COMPLETE")
print(json.dumps({
    "comparison_path": str(COMPARISON),
    "comparison_sha256": sha256(COMPARISON),
    "query_count": EXPECTED_QUERY_COUNT,
    "evaluation_executed": False,
    "batch_a_results_observed": False,
    "uses_mock": False,
}, ensure_ascii=False, indent=2))


In [ ]:
%%stage AG40
                                                                            
                                                                               

import hashlib
import json
import os
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

ROOT = Path("/content/reproduction_inputs/artifacts")
HISTORICAL_ROOT = ROOT / "evaluation/round1_method_lock"
LOCK_ROOT = ROOT / "evaluation/round2_evaluation"

METHOD_LOCK = HISTORICAL_ROOT / "ag19_region_comparison_v1/region_method_lock_v1.json"
EXTENSION_LOCK = LOCK_ROOT / "test48_batch_a_extension_lock_manifest.json"
EXTENSION = LOCK_ROOT / "batch_a_extension.json"
PROMPT_CATALOG = LOCK_ROOT / "prompt_centroid_v1/batch_a_extension_prompt_catalog.jsonl"
PROMPT_MANIFEST = LOCK_ROOT / "prompt_centroid_v1/batch_a_extension_prompt_catalog_manifest.json"
REGION_SELECTION = LOCK_ROOT / "ag39_batch_a_region_selection_v1/comparison.json"
OUTPUT = LOCK_ROOT / "ag40_batch_a_final_pre_evaluation_lock.json"

EXPECTED = {
    METHOD_LOCK: "8dc2e1e255f20f3e1289bca8bfe5fe4e79e3863e1d80710997aab621089d10bb",
    EXTENSION_LOCK: "46fccfad9678aaae2f4ee035c59fe1e60f6c49ec2b361dacc6b1e7a47fde14e9",
    PROMPT_CATALOG: "499cc0fb6f80223799a18999735c46e6331479147af64bca7815d80aaa0ad148",
    PROMPT_MANIFEST: "2aec23a3c9214e7dc60f3bf331bed4eb2d2869c01104cc2e6365205fab6543a0",
    REGION_SELECTION: "e34e4f6a0cf297e467d33f0c5dd6176bcbbe5b6140267ae1e1204d430d448b86",
}
EXPECTED_QUERY_COUNT = 93
STRATEGIES = [
    "target_centered",
    "highest_score_window",
    "minimum_unaffected_floor",
]


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical(value):
    return (
        json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
        + "\n"
    ).encode("utf-8")


for path, expected_hash in EXPECTED.items():
    require(path.is_file(), f"Missing required artifact: {path}")
    require(sha256(path) == expected_hash, f"SHA-256 drift: {path}")
require(EXTENSION.is_file(), f"Missing extension: {EXTENSION}")

method = json.loads(METHOD_LOCK.read_text(encoding="utf-8-sig"))
extension_lock = json.loads(EXTENSION_LOCK.read_text(encoding="utf-8-sig"))
extension = json.loads(EXTENSION.read_text(encoding="utf-8-sig"))
prompt_manifest = json.loads(PROMPT_MANIFEST.read_text(encoding="utf-8-sig"))
selection = json.loads(REGION_SELECTION.read_text(encoding="utf-8-sig"))
extension_hash = sha256(EXTENSION)

require(
    method.get("status") == "method_frozen_pending_evaluation_readiness"
    and method.get("selected_method") == "minimum_unaffected_floor",
    "Dev-selected method lock invalid",
)
require(method.get("baselines") == ["target_centered", "highest_score_window"],
        "Method baseline lock invalid")
require(
    extension_lock.get("status")
    == "test48_batch_a_extension_v2_locked_before_batch_a_attack"
    and extension_lock.get("locked_evaluation_ready") is True
    and extension_lock.get("evaluation_executed") is False,
    "AG37R v2 lock is not ready",
)
require(
    extension_lock.get("batch_a_extension", {}).get("sha256") == extension_hash,
    "AG37R extension binding drift",
)
require(
    prompt_manifest.get("status")
    == "batch_a_extension_prompt_catalog_locked_before_evaluation"
    and prompt_manifest.get("query_count") == EXPECTED_QUERY_COUNT
    and prompt_manifest.get("batch_a_results_observed") is False,
    "AG38 prompt lock invalid",
)
require(prompt_manifest.get("extension_sha256") == extension_hash,
        "AG38 extension binding drift")
require(
    selection.get("protocol", {}).get("scope")
    == "locked_batch_a_extension_cache_geometry_selection_not_attack_success",
    "AG39 selection scope invalid",
)
require(selection["protocol"].get("batch_a_results_observed") is False,
        "AG39 claims observed Batch A results")
require(selection["protocol"].get("extension_sha256") == extension_hash,
        "AG39 extension binding drift")
require(selection["protocol"].get("prompt_sha256") == EXPECTED[PROMPT_CATALOG],
        "AG39 prompt binding drift")
require(
    isinstance(extension, list)
    and len(extension) == EXPECTED_QUERY_COUNT
    and len({row.get("query_id") for row in extension}) == EXPECTED_QUERY_COUNT,
    "Expected 93 unique extension queries",
)
require(
    len({row.get("video_id") for row in extension}) == EXPECTED_QUERY_COUNT,
    "Batch A extension target videos are not distinct",
)

selection_rows = selection.get("results")
require(isinstance(selection_rows, list) and len(selection_rows) == EXPECTED_QUERY_COUNT,
        "AG39 selection count mismatch")
extension_by_id = {row["query_id"]: row for row in extension}
require({row.get("query_id") for row in selection_rows} == set(extension_by_id),
        "AG39 query set mismatch")
for row in selection_rows:
    query_id = row["query_id"]
    require(row.get("video_id") == extension_by_id[query_id].get("video_id"),
            f"{query_id}: AG39 target video drift")
    for strategy in STRATEGIES:
        positions = row.get("analysis", {}).get(strategy, {}).get("positions")
        require(
            isinstance(positions, list)
            and len(positions) == 16
            and positions == list(range(positions[0], positions[0] + 16)),
            f"{query_id}: invalid {strategy} interval",
        )

lock = {
    "schema_version": 1,
    "status": "batch_a_extension_final_pre_evaluation_configuration_locked",
    "artifact_class": "prospective_extension_pre_evaluation_lock",
    "method_lock_sha256": EXPECTED[METHOD_LOCK],
    "extension_lock_sha256": EXPECTED[EXTENSION_LOCK],
    "extension_sha256": extension_hash,
    "prompt_catalog_sha256": EXPECTED[PROMPT_CATALOG],
    "prompt_manifest_sha256": EXPECTED[PROMPT_MANIFEST],
    "region_selection_sha256": EXPECTED[REGION_SELECTION],
    "selected_method": "minimum_unaffected_floor",
    "baselines": ["target_centered", "highest_score_window"],
    "strategies": STRATEGIES,
    "query_count": EXPECTED_QUERY_COUNT,
    "video_count": EXPECTED_QUERY_COUNT,
    "region_budget": 16,
    "region_contiguous": True,
    "epsilon": 8 / 255,
    "alpha_start": 2 / 255,
    "alpha_end": 0.2 / 255,
    "steps": 15,
    "lambda1": 0.5,
    "lambda2": 0.0,
    "random_seed": 42,
    "window_size": 16,
    "stride": 1,
    "query_representation": "english_prompt_centroid",
    "historical_test48_results_observed": True,
    "batch_a_results_observed": False,
    "pooled_analysis_is_secondary": True,
    "uses_mock": False,
    "locked_evaluation_ready": True,
    "evaluation_executed": False,
    "evaluation_rule": (
        "Run all three locked strategies once on the 93-query Batch A extension; "
        "report every failure and paired denominator; do not tune after observation."
    ),
    "reporting_rule": (
        "Report historical organizer test48, prospective Batch A extension, and pooled "
        "results separately."
    ),
}
payload = canonical(lock)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
if OUTPUT.exists():
    require(OUTPUT.read_bytes() == payload, "Immutable AG40 lock drift")
else:
    temporary = OUTPUT.with_suffix(OUTPUT.suffix + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, OUTPUT)

print("AG40_BATCH_A_FINAL_PRE_EVALUATION_LOCK_OK")
print(json.dumps({
    **lock,
    "path": str(OUTPUT),
    "sha256": sha256(OUTPUT),
}, ensure_ascii=False, indent=2))


In [ ]:
%%stage AG41
                  
                                                                              
                                         
                                                                           
import gc, hashlib, importlib.metadata, json, os, random, shutil, subprocess, sys, time, urllib.request, zipfile
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def require(value, message):
    if not value:
        raise RuntimeError(message)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def canonical_bytes(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True,
                       separators=(",", ":")) + "\n").encode("utf-8")

def atomic_immutable(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        require(path.read_bytes() == payload, f"Immutable artifact drift: {path}")
        return
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, path)

STV_ROOT = Path("/content/reproduction_inputs/artifacts")
DATA_ROOT = Path("/content/reproduction_inputs/dataset")
LOCK_ROOT = STV_ROOT / "evaluation/round2_evaluation"
EXTENSION = LOCK_ROOT / "batch_a_extension.json"
EXTENSION_LOCK = LOCK_ROOT / "test48_batch_a_extension_lock_manifest.json"
PROMPT_CATALOG = LOCK_ROOT / "prompt_centroid_v1/batch_a_extension_prompt_catalog.jsonl"
PROMPT_MANIFEST = LOCK_ROOT / "prompt_centroid_v1/batch_a_extension_prompt_catalog_manifest.json"
REGION_SELECTION = LOCK_ROOT / "ag39_batch_a_region_selection_v1/comparison.json"
PRE_EVALUATION_LOCK = LOCK_ROOT / "ag40_batch_a_final_pre_evaluation_lock.json"
RECOVERY_ROOT = STV_ROOT / "frame_cache/repaired_frames/L22_V011"
CACHE_ROOT = STV_ROOT / "frame_cache/model_frame_cache"
OFFICIAL_DATASET_MANIFEST = RECOVERY_ROOT / "target_dataset_manifest_L22_V011_046_repair.json"
EXPECTED_VIDEO_IDS = STV_ROOT / "catalog/expected_video_ids.json"
CACHE_MANIFEST = CACHE_ROOT / "manifest.json"
CHECKPOINT = STV_ROOT / "artifacts/models/ViT-B-16-SigLIP_webli/open_clip_model.safetensors"
TOKENIZER_CACHE = STV_ROOT / "artifacts/models/ViT-B-16-SigLIP_webli/tokenizer_cache"
OUTPUT_ROOT = LOCK_ROOT / "ag41_batch_a_extension_region_evaluation_v2"
QUERIES = OUTPUT_ROOT / "batch_a_extension_queries.jsonl"
CONFIG_PATH = OUTPUT_ROOT / "centroid_t16_config.json"
AUTHORITY_PATH = OUTPUT_ROOT / "authority.jsonl"

EXPECTED_EXTENSION_LOCK_SHA = "46fccfad9678aaae2f4ee035c59fe1e60f6c49ec2b361dacc6b1e7a47fde14e9"
EXPECTED_EXTENSION_SHA = "c54db52c3c6427f11ef2af2e3e4242066b0d16bafcd4bd224b87a97a77f2e1a3"
EXPECTED_PROMPT_SHA = "499cc0fb6f80223799a18999735c46e6331479147af64bca7815d80aaa0ad148"
EXPECTED_PROMPT_MANIFEST_SHA = "2aec23a3c9214e7dc60f3bf331bed4eb2d2869c01104cc2e6365205fab6543a0"
EXPECTED_REGION_SELECTION_SHA = "e34e4f6a0cf297e467d33f0c5dd6176bcbbe5b6140267ae1e1204d430d448b86"
EXPECTED_PRE_EVALUATION_LOCK_SHA = "b6662e4bb08c37114f85acfc599887286fb6029a889a98fe530398140e27571c"
EXPECTED_DATASET_SHA = "9d3a579384bd8a94ab405a1bfbb39c065705140b593da310176f3f9c7d9351ac"
EXPECTED_VIDEO_IDS_SHA = "b47517d27e92f5b18d65579964c2cf8ea31454e3853b31aa939f3d2bb9948feb"
EXPECTED_CACHE_SHA = "5cc4b25610bd433a2abcc38a81dc234dc09bc1f83ecfc4a8dfe8b248e9c7b2a4"
EXPECTED_MODEL_SHA = "1d62e634a2ee5092794b3b3fbfeb07b0d9bdc3727b22d19a96ea2e25e654443a"
EXPECTED_CHECKPOINT_SHA = "81942ea8c09b9e41963357cca5d2682118bf7eb7491c2ae51a28bc4f4f95b194"

required_paths = [EXTENSION, EXTENSION_LOCK, PROMPT_CATALOG, PROMPT_MANIFEST,
                  REGION_SELECTION, PRE_EVALUATION_LOCK, DATA_ROOT,
                  OFFICIAL_DATASET_MANIFEST, EXPECTED_VIDEO_IDS,
                  CACHE_MANIFEST, CACHE_ROOT, CHECKPOINT, TOKENIZER_CACHE]
require(not [str(path) for path in required_paths if not path.exists()],
        f"Missing required assets: {[str(path) for path in required_paths if not path.exists()]}")
for path, expected, label in (
    (EXTENSION, EXPECTED_EXTENSION_SHA, "Batch A extension"),
    (EXTENSION_LOCK, EXPECTED_EXTENSION_LOCK_SHA, "extension lock"),
    (PROMPT_CATALOG, EXPECTED_PROMPT_SHA, "prompt catalog"),
    (PROMPT_MANIFEST, EXPECTED_PROMPT_MANIFEST_SHA, "prompt manifest"),
    (REGION_SELECTION, EXPECTED_REGION_SELECTION_SHA, "region selection"),
    (PRE_EVALUATION_LOCK, EXPECTED_PRE_EVALUATION_LOCK_SHA, "pre-evaluation lock"),
    (OFFICIAL_DATASET_MANIFEST, EXPECTED_DATASET_SHA, "dataset manifest"),
    (EXPECTED_VIDEO_IDS, EXPECTED_VIDEO_IDS_SHA, "expected video IDs"),
    (CACHE_MANIFEST, EXPECTED_CACHE_SHA, "cache manifest"),
    (CHECKPOINT, EXPECTED_CHECKPOINT_SHA, "checkpoint"),
):
    require(file_hash(path) == expected, f"{label} SHA-256 mismatch")

extension_lock = json.loads(EXTENSION_LOCK.read_bytes())
prompt_manifest = json.loads(PROMPT_MANIFEST.read_bytes())
pre_evaluation_lock = json.loads(PRE_EVALUATION_LOCK.read_bytes())
selected = sorted(json.loads(EXTENSION.read_bytes()), key=lambda row: row["query_id"])
require(len(selected) == 93 and len({row["query_id"] for row in selected}) == 93,
        "Expected 93 unique Batch A extension queries")
require(extension_lock.get("status") == "test48_batch_a_extension_v2_locked_before_batch_a_attack"
        and extension_lock.get("batch_a_extension", {}).get("sha256") == EXPECTED_EXTENSION_SHA,
        "AG37R v2 extension lock drift")
require(prompt_manifest.get("status") == "batch_a_extension_prompt_catalog_locked_before_evaluation"
        and prompt_manifest.get("prompt_catalog_sha256") == EXPECTED_PROMPT_SHA,
        "AG38 prompt lock drift")
require(pre_evaluation_lock.get("status") == "batch_a_extension_final_pre_evaluation_configuration_locked"
        and pre_evaluation_lock.get("locked_evaluation_ready") is True
        and pre_evaluation_lock.get("evaluation_executed") is False
        and pre_evaluation_lock.get("batch_a_results_observed") is False,
        "AG40 pre-evaluation lock is not ready")

try:
    version = importlib.metadata.version("open_clip_torch")
except importlib.metadata.PackageNotFoundError:
    version = None
if version != "3.3.0":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch==3.3.0"])

RUNTIME_ROOT = Path("/content/stv_ag41_runtime")
runtime_zip = Path("/content/stv_ag41_runtime.zip")
token = userdata.get("GITHUB_TOKEN")
require(isinstance(token, str) and token.strip(), "Add GITHUB_TOKEN to Colab Secrets")
if RUNTIME_ROOT.exists(): shutil.rmtree(RUNTIME_ROOT)
RUNTIME_ROOT.mkdir(parents=True)
request = urllib.request.Request(
    "https://api.github.com/repos/haruxne/STV-Attack/zipball/ab526c50f491bae440ea697fca72adcb10c687f3",
    headers={"Authorization": f"Bearer {token.strip()}", "Accept": "application/vnd.github+json"})
with urllib.request.urlopen(request, timeout=120) as response:
    runtime_zip.write_bytes(response.read())
with zipfile.ZipFile(runtime_zip) as archive: archive.extractall(RUNTIME_ROOT)
roots = [path for path in RUNTIME_ROOT.iterdir() if path.is_dir()]
require(len(roots) == 1 and (roots[0] / "src").is_dir(), "Unexpected repository archive")
RUNTIME_ROOT = roots[0]
sys.path.insert(0, str(RUNTIME_ROOT))
for _name in list(sys.modules):
    if _name == "src" or _name.startswith("src."): del sys.modules[_name]
import torch
from src.clean_runner import CleanConfig, _load_and_validate_config as load_clean_config, run_clean_retrieval
from src.experiment_runner import AttackConfig, _load_and_validate_config as load_attack_config, run_attack_experiment
from src.result_authority import load_result_authority
from src.statistics import build_statistical_report

require(torch.cuda.is_available(), "AG41 requires CUDA")
print("AG41_GPU", torch.cuda.get_device_name(0), flush=True)
gc.collect()
torch.cuda.empty_cache()
free_gpu, total_gpu = torch.cuda.mem_get_info()
print("AG41_GPU_FREE_GIB", round(free_gpu / 2**30, 2), "OF", round(total_gpu / 2**30, 2), flush=True)
require(
    free_gpu >= 18 * 2**30,
    "GPU already contains live tensors/models. Restart the Colab session, then run only this AG41 cell.",
)

                                                                             
                                                                     
atomic_immutable(QUERIES, b"".join(canonical_bytes(row) for row in selected))
queries_hash = file_hash(QUERIES)

window_config = {
    "window_size": 16,
    "stride": 1,
    "query_text_field": "query_text_en",
    "query_representation": {
        "mode": "english_prompt_centroid",
        "catalog_path": str(PROMPT_CATALOG),
        "catalog_sha256": EXPECTED_PROMPT_SHA,
    },
}
atomic_immutable(CONFIG_PATH, canonical_bytes(window_config))
_, clean_hash = load_clean_config(CONFIG_PATH)
common = dict(
    queries=QUERIES, expected_query_sha256=queries_hash,
    split=EXTENSION, expected_split_sha256=EXPECTED_EXTENSION_SHA,
    cache_manifest=CACHE_MANIFEST, expected_cache_manifest_sha256=EXPECTED_CACHE_SHA,
    cache_root=CACHE_ROOT, data_root=DATA_ROOT,
    dataset_hash=EXPECTED_DATASET_SHA, model_hash=EXPECTED_MODEL_SHA,
    checkpoint_path=CHECKPOINT, expected_checkpoint_sha256=EXPECTED_CHECKPOINT_SHA,
    tokenizer_cache_dir=TOKENIZER_CACHE, config=CONFIG_PATH,
    authority_output=AUTHORITY_PATH, device="cuda", resume=True,
    expected_video_ids=EXPECTED_VIDEO_IDS, expected_video_ids_sha256=EXPECTED_VIDEO_IDS_SHA,
    official_dataset_manifest=OFFICIAL_DATASET_MANIFEST,
)



import inspect
import random
import numpy as np
import src.experiment_runner as runner

                                                                        
                                                                            
                                                                           
source_by_id = {row["query_id"]: row for row in selected}
require(len(source_by_id) == 93, "Batch A extension source is incomplete")


def validate_extension_execution_annotations(records, *, data_root,
        official_dataset_manifest, expected_manifest_sha256, locked=True):
    errors=[]
    if locked is not True:
        errors.append({'line':0,'field':'locked','message':'extension evaluation requires locked=True'})
    if file_hash(official_dataset_manifest)!=expected_manifest_sha256:
        errors.append({'line':0,'field':'official_dataset_manifest','message':'manifest hash drift'})
    manifest=json.loads(official_dataset_manifest.read_bytes())
    videos={v['video_id']:v for v in manifest.get('videos',[])}
    seen=set()
    for line,row in enumerate(records,1):
        qid=row.get('query_id'); source=source_by_id.get(qid)
        if source is None or qid in seen:
            errors.append({'line':line,'field':'query_id','message':'missing/duplicate locked extension ID'});continue
        seen.add(qid)
        for field in (
            'query_text_vi','query_text_en','query_text_sha256',
            'query_text_vi_sha256','query_text_en_sha256',
            'retrieval_query_text_sha256','video_id','target_keyframe_n',
            'target_frame_id','target_provenance','batch_a_source_query_id',
            'batch_a_source_candidate_id','batch_a_source_record_sha256',
        ):
            if row.get(field)!=source.get(field):
                errors.append({'line':line,'field':field,
                               'message':'differs from hash-bound AG37R extension source'})
        if source.get('unique_match_pass') is not True:
            errors.append({'line':line,'field':'unique_match_pass',
                           'message':'AGB4 uniqueness promotion missing'})
        if source.get('locked_evaluation_ready') is not True:
            errors.append({'line':line,'field':'locked_evaluation_ready',
                           'message':'AG37R record lock missing'})
        review=source.get('human_uniqueness_review')
        if not isinstance(review,dict) or review.get('decision')!='pass':
            errors.append({'line':line,'field':'human_uniqueness_review',
                           'message':'AGB3R pass evidence missing'})
        video=videos.get(row.get('video_id'))
        if video is None:
            errors.append({'line':line,'field':'video_id','message':'absent from official manifest'})
        else:
            provenance=source.get('target_provenance')
            mapping=video.get('mapping')
            if not isinstance(provenance,dict) or not isinstance(mapping,dict) or \
                    provenance.get('mapping_sha256')!=mapping.get('sha256'):
                errors.append({'line':line,'field':'target_provenance',
                               'message':'official mapping binding mismatch'})
    if seen != set(source_by_id):
        errors.append({'line':0,'field':'query_id','message':'extension query set incomplete'})
    return {'schema_version':1,'status':'valid' if not errors else 'invalid','locked':True,
            'record_count':len(records),'distinct_video_count':len({r.get('video_id') for r in records}),
            'validator':'agb4_ag37r_extension_hash_binding_v1','errors':errors,
            'warnings':['AGB5 hard-negative and delayed self-audit gates were not performed; '
                        'evaluation scope follows the AG40 revised prospective-extension lock.']}

require(Path(inspect.getfile(runner)).resolve().is_relative_to(RUNTIME_ROOT.resolve()),
        "Unexpected runner import")
split_path = OUTPUT_ROOT / "all_batch_a_extension.json"
atomic_immutable(split_path, canonical_bytes(selected))
common.update(split=split_path, expected_split_sha256=file_hash(split_path))

original = runner.run_pgd_attack

def memory_safe_scheduled_attack(adapter, video_frames, query_embedding, config,
                                 window_size, stride, changed_positions=None):
    """Exact locked PGD objective with only the editable interval resident on CUDA."""
    device = torch.device(config.device)
    require(device.type == "cuda", "AG41 memory-safe runner requires CUDA")
    require(query_embedding.ndim == 1, "query_embedding must be one-dimensional")
    query = query_embedding.detach().to(device)
    frame_count = int(video_frames.shape[0])
    all_windows = runner.build_sliding_windows(frame_count, window_size=window_size, stride=stride)
    changed = tuple(range(frame_count)) if changed_positions is None else tuple(sorted(set(changed_positions)))
    require(changed and all(isinstance(p, int) and not isinstance(p, bool) and 0 <= p < frame_count for p in changed),
            "changed_positions must identify original video frames")
    require(changed == tuple(range(changed[0], changed[0] + len(changed))),
            "AG41 requires one contiguous editable interval")
    affected_indices = runner.overlapping_window_indices(
        frame_count, changed, window_size=window_size, stride=stride)
    windows = tuple(all_windows[i] for i in affected_indices)
    changed_lookup = {position: offset for offset, position in enumerate(changed)}
    clean_changed = video_frames[list(changed)].detach().to(device)

    def temporal_losses_exact(delta):
        pixels = delta.shape[1] * delta.shape[2] * delta.shape[3]
        zero = delta.new_zeros(())
        first_sum = (delta[1:] - delta[:-1]).square().sum() if len(changed) > 1 else zero
        if changed[0] > 0:
            first_sum = first_sum + delta[0].square().sum()
        if changed[-1] < frame_count - 1:
            first_sum = first_sum + delta[-1].square().sum()
        flicker = first_sum / max((frame_count - 1) * pixels, 1)

        accel_sum = zero
        if frame_count > 2:
            for center in range(max(0, changed[0] - 2), min(frame_count - 3, changed[-1]) + 1):
                terms = []
                for position, coefficient in ((center, 1.0), (center + 1, -2.0), (center + 2, 1.0)):
                    offset = changed_lookup.get(position)
                    if offset is not None:
                        terms.append(coefficient * delta[offset])
                if terms:
                    curvature = terms[0]
                    for term in terms[1:]:
                        curvature = curvature + term
                    accel_sum = accel_sum + curvature.square().sum()
        acceleration = accel_sum / max((frame_count - 2) * pixels, 1)
        return flicker, acceleration

    def clip_for(window, delta):
        frames = []
        for position in window:
            offset = changed_lookup.get(position)
            if offset is None:
                frames.append(video_frames[position].to(device))
            else:
                frames.append(torch.clamp(clean_changed[offset] + delta[offset], 0.0, 1.0))
        return torch.stack(frames).unsqueeze(0)

    def streaming_max_similarity(delta, *, gradients):
        context = torch.enable_grad() if gradients else torch.no_grad()
        best = None
        with context:
            for window in windows:
                clip = clip_for(window, delta)
                score = torch.mv(adapter.get_video_embedding(clip), query).squeeze(0)
                if best is None or float(score.detach()) > float(best.detach()):
                    best = score
                del clip
        require(best is not None, "No affected retrieval window")
        return best

    if config.condition == "random":
        generator = torch.Generator(device=device)
        generator.manual_seed(config.random_seed)
        delta = torch.empty_like(clean_changed).uniform_(-config.epsilon, config.epsilon,
                                                          generator=generator)
        delta = torch.maximum(torch.minimum(delta, 1.0 - clean_changed), -clean_changed)
        iterations = 1
    else:
        delta = torch.zeros_like(clean_changed, requires_grad=True)
        for step in range(config.steps):
            sim_loss = streaming_max_similarity(delta, gradients=True)
            flicker_loss, accel_loss = temporal_losses_exact(delta)
            loss = sim_loss + config.lambda1 * flicker_loss + config.lambda2 * accel_loss
            loss.backward()
            require(delta.grad is not None, "PGD produced no perturbation gradient")
            alpha = config.alpha * (1.0 - 0.9 * step / max(config.steps - 1, 1))
            with torch.no_grad():
                delta.sub_(alpha * delta.grad.sign())
                delta.clamp_(-config.epsilon, config.epsilon)
                delta.copy_(torch.maximum(torch.minimum(delta, 1.0 - clean_changed), -clean_changed))
            delta.grad = None
        iterations = config.steps

    with torch.no_grad():
        adversarial_changed = torch.clamp(clean_changed + delta.detach(), 0.0, 1.0)
        sim_loss = streaming_max_similarity(delta.detach(), gradients=False)
        flicker_loss, accel_loss = temporal_losses_exact(delta.detach())
        squared_error = (adversarial_changed - clean_changed).square().sum().item()
        mse = squared_error / video_frames.numel()
        psnr = float("inf") if mse == 0 else 10.0 * __import__("math").log10(1.0 / mse)
        changed_ssim = sum(
            runner.compute_ssim(clean_changed[i:i + 1], adversarial_changed[i:i + 1])
            for i in range(len(changed))
        ) / len(changed)
        ssim = ((frame_count - len(changed)) + len(changed) * changed_ssim) / frame_count
        linf = delta.detach().abs().max().item()
        adversarial_cpu = adversarial_changed.cpu()

    # Reuse the already-loaded CPU tensor; only the locked 16-frame interval changes.
    video_frames[list(changed)] = adversarial_cpu
    metrics = {
        "psnr": psnr if __import__("math").isfinite(psnr) else None,
        "psnr_reason": "identical tensors" if not __import__("math").isfinite(psnr) else None,
        "ssim": float(ssim), "perturbation_linf": linf,
        "flicker": flicker_loss.item(), "acceleration": accel_loss.item(),
        "loss_sim": sim_loss.item(), "iterations": iterations,
        "changed_frame_count": len(changed),
        "affected_window_count": len(affected_indices),
    }
    del clean_changed, delta, adversarial_changed, adversarial_cpu, query
    torch.cuda.empty_cache()
    return video_frames, metrics

scheduled = memory_safe_scheduled_attack
scheduled_source = inspect.getsource(memory_safe_scheduled_attack)

require(len(selected)==93, 'Expected locked Batch A extension')
region_selection=json.loads(REGION_SELECTION.read_bytes())
require(file_hash(REGION_SELECTION)==EXPECTED_REGION_SELECTION_SHA,
        'AG39 comparison hash drift')
require(region_selection['protocol']['extension_sha256']==EXPECTED_EXTENSION_SHA and
        region_selection['protocol']['cache_sha256']==EXPECTED_CACHE_SHA and
        region_selection['protocol']['prompt_sha256']==EXPECTED_PROMPT_SHA and
        region_selection['protocol']['budget']==16 and
        region_selection['protocol']['contiguous_only'] is True and
        region_selection['protocol']['batch_a_results_observed'] is False,
        'AG39 input/protocol mismatch')
require(file_hash(PRE_EVALUATION_LOCK)==EXPECTED_PRE_EVALUATION_LOCK_SHA,
        'AG40 lock hash drift')
require(pre_evaluation_lock.get('extension_sha256')==EXPECTED_EXTENSION_SHA and
        pre_evaluation_lock.get('region_selection_sha256')==EXPECTED_REGION_SELECTION_SHA and
        pre_evaluation_lock.get('query_count')==93,
        'AG40 extension lock mismatch')
selection_by_query={r['query_id']:r for r in region_selection['results']}
require(len(selection_by_query)==93 and set(selection_by_query)=={r['query_id'] for r in selected},
        'AG39 query set mismatch')
strategies=('target_centered','highest_score_window','minimum_unaffected_floor')
for record in selected:
    saved=selection_by_query[record['query_id']]
    require(saved['video_id']==record['video_id'], 'AG39 target mismatch')
    for strategy in strategies:
        positions=saved['analysis'][strategy]['positions']
        require(len(positions)==16 and positions==list(range(positions[0],positions[0]+16)),
                'Invalid contiguous 16-frame selection')

                                                                              
                                                                    
original_run=runner.run_attack_experiment
run_source=inspect.getsource(original_run)
                                                                           
                                                                              
legacy_keyframe='target_keyframe_n = record.get("keyframe_n")'
require(run_source.count(legacy_keyframe)==1, 'Pinned runner keyframe marker changed')
run_source=run_source.replace(
    legacy_keyframe,
    'target_keyframe_n = record.get("target_keyframe_n")',
)
marker='            # 4. Execute attack'
require(run_source.count(marker)==1, 'Runner source changed')
insertion="""        # AG41: frozen cache-derived selection, same contiguous frame budget.
        chosen = region_selections[query_id]["analysis"][region_strategy]["positions"]
        if region_strategy == "target_centered":
            if list(changed_positions) != chosen:
                raise ExperimentRunnerError("AG39 target-centered baseline drift")
        changed_positions = tuple(chosen)
        if any(p < 0 or p >= video_frames.shape[0] for p in changed_positions):
            raise ExperimentRunnerError("Region outside target video")

"""
insertion='\n'.join('    '+line if line else line for line in insertion.split('\n'))
modified_source=run_source.replace(marker,insertion+marker)
oom_handler='''        except Exception as e:
            logger.error(f"Failure on {query_id}: {e}")'''
require(modified_source.count(oom_handler)==1, 'Runner exception handler changed')
modified_source=modified_source.replace(oom_handler, '''        except Exception as e:
            if isinstance(e, torch.OutOfMemoryError):
                raise
            logger.error(f"Failure on {query_id}: {e}")''')
protocol={'scope':'locked_batch_a_extension_final_region_evaluation_once_only_v2',
    'historical_test48_results_observed':True,'batch_a_results_observed_before_run':False,
    'supersedes':'ag41_batch_a_extension_region_evaluation_v1_preflight_failed_before_attack',
    'validator_aliases':['validate_annotations','validate_execution_annotations'],
    'strategies':strategies,'ag39_comparison_sha256':file_hash(REGION_SELECTION),
    'ag40_lock_sha256':file_hash(PRE_EVALUATION_LOCK),
    'extension_source_sha256':EXPECTED_EXTENSION_SHA,'query_jsonl_sha256':queries_hash,
    'extension_validator_sha256':hashlib.sha256(inspect.getsource(validate_extension_execution_annotations).encode()).hexdigest(),
    'scheduled_function_sha256':hashlib.sha256(scheduled_source.encode()).hexdigest(),
    'region_runner_sha256':hashlib.sha256(modified_source.encode()).hexdigest(),
    'epsilon':8/255,'steps':15,'lambda1':0.5,'lambda2':0.,
    'schedule':'linear alpha 2/255 to 0.2/255', 'frame_budget':16,
    'attack_memory_plan':'streaming exact max-gradient; only the locked editable interval resides persistently on CUDA',
    'threat_model':'attacker may select any single contiguous interval of 16 sampled frames',
    'identity':'protocol hash and strategy accompany original config hash; config hash alone omits selector/schedule',
    'reporting':'All 93 paired ranks, scores, distortion and runtime; report Batch A separately from historical test48 and compare to highest-score baseline.'}
atomic_immutable(OUTPUT_ROOT/'protocol.json',canonical_bytes(protocol))
protocol_hash=file_hash(OUTPUT_ROOT/'protocol.json')
print('AG41_START',protocol,flush=True)
reports=[]
try:
    for strategy in strategies:
        namespace=dict(runner.__dict__)
        namespace.update(run_pgd_attack=scheduled, region_strategy=strategy,
                         region_selections=selection_by_query,
                         validate_annotations=validate_extension_execution_annotations,
                         validate_execution_annotations=validate_extension_execution_annotations)
        exec(compile(modified_source,'<ag41-region-runner>','exec'),namespace)
        folder=OUTPUT_ROOT/strategy
        identity={'protocol_sha256':protocol_hash,'strategy':strategy,
                  'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0)}
        atomic_immutable(folder/'run_identity.json',canonical_bytes(identity))
        random.seed(42)
        np.random.seed(42)
        torch.manual_seed(42)
        torch.cuda.manual_seed_all(42)
        torch.backends.cudnn.benchmark=False
        path=folder/'authority.jsonl'
        print('AG41_STRATEGY_START',strategy,flush=True)
        namespace['run_attack_experiment'](AttackConfig(**{**common,'authority_output':path},
            condition='first_order',epsilon=8/255,alpha=2/255,steps=15,lambda1=0.5,lambda2=0.,random_seed=42))
        rows=load_result_authority(path)
        require(len(rows)==93 and {r['query_id'] for r in rows}==set(selection_by_query)
                and all(r['status']=='success' for r in rows),'Incomplete/failed strategy; rerun to resume')
        for r in rows:
            require(r['measurements']['changed_positions_zero_based']==
                    selection_by_query[r['query_id']]['analysis'][strategy]['positions'],
                    'Recorded edit positions mismatch')
        report={**identity,'authority_sha256':file_hash(path),'rows':rows}
        reports.append(report)
        print('AG41_STRATEGY_COMPLETE',strategy,flush=True)
        gc.collect()
        torch.cuda.empty_cache()
finally:
    runner.run_pgd_attack=original
by_strategy={r['strategy']:{x['query_id']:x for x in r['rows']} for r in reports}
paired=[]
for qid in sorted(selection_by_query):
    row={'query_id':qid}
    for strategy in strategies:
        r=by_strategy[strategy][qid]
        row[strategy]={'rank':r['rank'], **{k:r['measurements'][k] for k in
            ('score','clean_rank','psnr','ssim','flicker','acceleration','elapsed_time')}}
    paired.append(row)
summary={}
for baseline in strategies[:2]:
    differences=[r['minimum_unaffected_floor']['rank']-r[baseline]['rank'] for r in paired]
    summary[baseline]={'rank_better_equal_worse':[sum(d>0 for d in differences),
        sum(d==0 for d in differences),sum(d<0 for d in differences)],
        'mean_rank_difference':sum(differences)/93}
atomic_immutable(OUTPUT_ROOT/'comparison.json',canonical_bytes(
    {'protocol':protocol,'results':reports,'paired':paired,'summary':summary}))
print('AG41_PAIRED',json.dumps(paired),flush=True)
print('AG41_SUMMARY',json.dumps(summary),flush=True)
print('AG41_BATCH_A_EXTENSION_EVALUATION_COMPLETE',json.dumps({
    'comparison_path':str(OUTPUT_ROOT/'comparison.json'),
    'comparison_sha256':file_hash(OUTPUT_ROOT/'comparison.json'),
    'query_count':93,'strategy_count':3,'execution_count':279,
    'uses_mock':False},ensure_ascii=False),flush=True)


In [ ]:
%%stage AG42
"""AG42: CPU-only audit of Batch A and secondary pooled paper inputs.

Run after AG41 prints AG41_BATCH_A_EXTENSION_EVALUATION_COMPLETE. This cell
performs no inference and does not modify any locked input or attack result.
"""
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path
import hashlib
import json
import math
import os

import numpy as np


def require(ok, message):
    if not ok:
        raise RuntimeError(message)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_bytes(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True,
                       separators=(",", ":")) + "\n").encode("utf-8")


def finite_number(value):
    return (isinstance(value, (int, float)) and not isinstance(value, bool)
            and math.isfinite(value))


def exact_two_sided_sign_test(wins, losses):
    non_ties = wins + losses
    if non_ties == 0:
        return None
    tail = min(wins, losses)
    probability = 2.0 * sum(
        math.comb(non_ties, k) for k in range(tail + 1)
    ) / (2 ** non_ties)
    return min(1.0, probability)


ROOT = Path("/content/reproduction_inputs/artifacts")
BATCH_ROOT = ROOT / "evaluation/round2_evaluation"
BATCH_RUN = BATCH_ROOT / "ag41_batch_a_extension_region_evaluation_v2"
BATCH_SOURCE = BATCH_ROOT / "batch_a_extension.json"
BATCH_LOCK = BATCH_ROOT / "test48_batch_a_extension_lock_manifest.json"
BATCH_PROMPTS = BATCH_ROOT / "prompt_centroid_v1/batch_a_extension_prompt_catalog.jsonl"
BATCH_PROMPT_MANIFEST = BATCH_ROOT / "prompt_centroid_v1/batch_a_extension_prompt_catalog_manifest.json"
BATCH_SELECTION = BATCH_ROOT / "ag39_batch_a_region_selection_v1/comparison.json"
BATCH_PRELOCK = BATCH_ROOT / "ag40_batch_a_final_pre_evaluation_lock.json"
BATCH_PROTOCOL = BATCH_RUN / "protocol.json"
BATCH_COMPARISON = BATCH_RUN / "comparison.json"

HIST_ROOT = ROOT / "evaluation/round1_evaluation"
HIST_RUN = HIST_ROOT / "ag35_final_test48_region_evaluation_v5"
HIST_SOURCE = HIST_ROOT / "test48.json"
HIST_COMPARISON = HIST_RUN / "comparison.json"
HIST_AUDIT = HIST_ROOT / "ag36_test48_cpu_result_audit_v1.json"

OUTPUT = BATCH_ROOT / "ag42_batch_a_and_pooled_cpu_result_audit_v1.json"

EXPECTED = {
    BATCH_SOURCE: "c54db52c3c6427f11ef2af2e3e4242066b0d16bafcd4bd224b87a97a77f2e1a3",
    BATCH_LOCK: "46fccfad9678aaae2f4ee035c59fe1e60f6c49ec2b361dacc6b1e7a47fde14e9",
    BATCH_PROMPTS: "499cc0fb6f80223799a18999735c46e6331479147af64bca7815d80aaa0ad148",
    BATCH_PROMPT_MANIFEST: "2aec23a3c9214e7dc60f3bf331bed4eb2d2869c01104cc2e6365205fab6543a0",
    BATCH_SELECTION: "e34e4f6a0cf297e467d33f0c5dd6176bcbbe5b6140267ae1e1204d430d448b86",
    BATCH_PRELOCK: "b6662e4bb08c37114f85acfc599887286fb6029a889a98fe530398140e27571c",
    BATCH_COMPARISON: "751d9837819836fe0643fa1bd17af93cb979aa2e096fdd18effd8fc0b6c721b6",
    HIST_SOURCE: "231ff1f0432be2bbc1eee9f879a5284f3d6c7633c5c674fdabd605495fd607f5",
    HIST_COMPARISON: "e688273cb949b53728d5434d4426f50320935004baa22fa82e58db9323689a15",
    HIST_AUDIT: "a82b50ff4b32e3f8bd37dd03ca3210490266060bc635b441029c5a697786ca81",
}
for path, expected_hash in EXPECTED.items():
    require(path.is_file(), f"Missing required artifact: {path}")
    require(sha256(path) == expected_hash, f"SHA-256 drift: {path}")
require(BATCH_PROTOCOL.is_file(), "AG41 protocol.json is missing")

strategies = ["target_centered", "highest_score_window", "minimum_unaffected_floor"]
selected_strategy = "minimum_unaffected_floor"

batch_source = json.loads(BATCH_SOURCE.read_bytes())
hist_source = json.loads(HIST_SOURCE.read_bytes())
batch_selection = json.loads(BATCH_SELECTION.read_bytes())
batch_prelock = json.loads(BATCH_PRELOCK.read_bytes())
batch_protocol = json.loads(BATCH_PROTOCOL.read_bytes())
batch_comparison = json.loads(BATCH_COMPARISON.read_bytes())
hist_comparison = json.loads(HIST_COMPARISON.read_bytes())
hist_audit = json.loads(HIST_AUDIT.read_bytes())

require(len(batch_source) == 93 and len(hist_source) == 48,
        "Cohort query count mismatch")
require(len({row["query_id"] for row in batch_source}) == 93,
        "Duplicate Batch A query ID")
require(len({row["query_id"] for row in hist_source}) == 48,
        "Duplicate historical query ID")
require(batch_prelock.get("status") ==
        "batch_a_extension_final_pre_evaluation_configuration_locked",
        "AG40 status drift")
require(batch_prelock.get("evaluation_executed") is False and
        batch_prelock.get("batch_a_results_observed") is False,
        "AG40 is not a clean pre-evaluation lock")
require(batch_protocol.get("scope") ==
        "locked_batch_a_extension_final_region_evaluation_once_only_v2",
        "AG41 protocol scope drift")
require(list(batch_protocol.get("strategies", [])) == strategies,
        "AG41 strategy order drift")
require(batch_protocol.get("ag39_comparison_sha256") == EXPECTED[BATCH_SELECTION],
        "AG41/AG39 binding drift")
require(batch_protocol.get("ag40_lock_sha256") == EXPECTED[BATCH_PRELOCK],
        "AG41/AG40 binding drift")
require(batch_protocol.get("extension_source_sha256") == EXPECTED[BATCH_SOURCE],
        "AG41 extension binding drift")
for key, expected_value in {
    "epsilon": 8 / 255,
    "steps": 15,
    "lambda1": 0.5,
    "lambda2": 0.0,
    "frame_budget": 16,
}.items():
    require(batch_protocol.get(key) == expected_value,
            f"AG41 locked parameter drift: {key}")
require(batch_comparison.get("protocol") == batch_protocol,
        "AG41 comparison protocol drift")


def verify_run(label, run_root, comparison, source, selection=None):
    source_by_id = {row["query_id"]: row for row in source}
    query_ids = [row["query_id"] for row in source]
    paired_rows = comparison.get("paired", [])
    paired_by_id = {row.get("query_id"): row for row in paired_rows}
    require(len(paired_rows) == len(source) and set(paired_by_id) == set(query_ids),
            f"{label}: paired query set mismatch")
    reports = comparison.get("results", [])
    require(len(reports) == 3 and
            [report.get("strategy") for report in reports] == strategies,
            f"{label}: three-strategy report mismatch")
    selection_by_id = None
    if selection is not None:
        selection_rows = selection.get("results", [])
        selection_by_id = {row.get("query_id"): row for row in selection_rows}
        require(len(selection_rows) == len(source) and
                set(selection_by_id) == set(query_ids),
                f"{label}: region-selection query set mismatch")

    authority_hashes = {}
    identities = {}
    config_hashes = set()
    for report in reports:
        strategy = report["strategy"]
        authority = run_root / strategy / "authority.jsonl"
        identity_path = run_root / strategy / "run_identity.json"
        require(authority.is_file() and identity_path.is_file(),
                f"{label}: missing {strategy} authority or identity")
        authority_hash = sha256(authority)
        require(authority_hash == report.get("authority_sha256"),
                f"{label}: {strategy} authority hash drift")
        identity = json.loads(identity_path.read_bytes())
        require(identity.get("protocol_sha256") == sha256(run_root / "protocol.json"),
                f"{label}: {strategy} protocol identity drift")
        require(identity.get("strategy") == strategy,
                f"{label}: {strategy} identity drift")
        for key in ("torch", "cuda", "gpu"):
            require(report.get(key) == identity.get(key),
                    f"{label}: {strategy}/{key} identity mismatch")
        rows = [json.loads(line) for line in authority.read_text(
            encoding="utf-8").splitlines() if line.strip()]
        rows_by_id = {row.get("query_id"): row for row in rows}
        require(len(rows) == len(source) and set(rows_by_id) == set(query_ids),
                f"{label}: {strategy} authority query set mismatch")
        require(report.get("rows") == rows,
                f"{label}: {strategy} embedded authority drift")
        for query_id in query_ids:
            row = rows_by_id[query_id]
            expected = source_by_id[query_id]
            require(row.get("status") == "success" and
                    row.get("condition") == "first_order",
                    f"{label}: {strategy}/{query_id} failed or condition drifted")
            require(row.get("video_id") == expected.get("video_id"),
                    f"{label}: {strategy}/{query_id} target mismatch")
            config_hashes.add(row.get("config_hash"))
            measurements = row.get("measurements", {})
            require(measurements.get("uses_mock") is False and
                    measurements.get("device") == "cuda",
                    f"{label}: {strategy}/{query_id} lacks real CUDA evidence")
            require(measurements.get("changed_frame_count") == 16,
                    f"{label}: {strategy}/{query_id} frame budget drift")
            require(measurements.get("perturbation_linf", float("inf")) <=
                    8 / 255 + 1e-5,
                    f"{label}: {strategy}/{query_id} epsilon violation")
            if selection_by_id is not None:
                expected_positions = selection_by_id[query_id]["analysis"][strategy]["positions"]
                require(measurements.get("changed_positions_zero_based") == expected_positions,
                        f"{label}: {strategy}/{query_id} interval mismatch")
            require(isinstance(row.get("rank"), int) and row["rank"] >= 1,
                    f"{label}: {strategy}/{query_id} invalid rank")
            for key in ("score", "clean_rank", "ssim", "flicker",
                        "acceleration", "elapsed_time"):
                require(finite_number(measurements.get(key)),
                        f"{label}: {strategy}/{query_id} invalid {key}")
            require(measurements.get("psnr") is None or
                    finite_number(measurements.get("psnr")),
                    f"{label}: {strategy}/{query_id} invalid psnr")
            paired = paired_by_id[query_id][strategy]
            require(row["rank"] == paired.get("rank"),
                    f"{label}: {strategy}/{query_id} paired rank drift")
            for key in ("score", "clean_rank", "psnr", "ssim", "flicker",
                        "acceleration", "elapsed_time"):
                require(measurements.get(key) == paired.get(key),
                        f"{label}: {strategy}/{query_id} paired {key} drift")
        authority_hashes[strategy] = authority_hash
        identities[strategy] = identity
    require(len(config_hashes) == 1 and None not in config_hashes,
            f"{label}: config hash is not uniform")
    for query_id in query_ids:
        clean = [paired_by_id[query_id][strategy]["clean_rank"]
                 for strategy in strategies]
        require(clean[1:] == clean[:-1],
                f"{label}: {query_id} clean-rank parity failed")
    return {
        "query_ids": query_ids,
        "paired_by_id": paired_by_id,
        "video_by_id": {row["query_id"]: row["video_id"] for row in source},
        "authority_hashes": authority_hashes,
        "runtime_identities": identities,
        "config_hash": next(iter(config_hashes)),
    }


batch_verified = verify_run(
    "Batch A", BATCH_RUN, batch_comparison, batch_source, batch_selection)
hist_verified = verify_run(
    "historical test48", HIST_RUN, hist_comparison, hist_source)


def summarize(query_ids, paired_by_id, video_by_id, seed):
    clean_ranks = np.array([
        paired_by_id[q][strategies[0]]["clean_rank"] for q in query_ids
    ], dtype=int)
    ranks = {
        strategy: np.array([
            paired_by_id[q][strategy]["rank"] for q in query_ids
        ], dtype=float)
        for strategy in strategies
    }
    video_ids = sorted({video_by_id[q] for q in query_ids})
    groups = [np.array([
        index for index, query_id in enumerate(query_ids)
        if video_by_id[query_id] == video_id
    ]) for video_id in video_ids]
    rng = np.random.default_rng(seed)
    bootstrap_samples = [
        np.concatenate([groups[index] for index in
                        rng.integers(len(groups), size=len(groups))])
        for _ in range(10000)
    ]
    result = {
        "query_count": len(query_ids),
        "video_count": len(video_ids),
        "paired_denominator": len(query_ids),
        "bootstrap": {
            "method": "paired video-cluster bootstrap",
            "cluster_count": len(video_ids),
            "query_weighted": True,
            "draws": 10000,
            "seed": seed,
            "interval": "percentile 95%",
        },
        "metrics": {},
        "paired_comparisons": {},
    }
    for strategy in strategies:
        values = ranks[strategy]
        eligible10 = clean_ranks <= 10
        shift = values - clean_ranks
        psnr = [paired_by_id[q][strategy]["psnr"] for q in query_ids
                if paired_by_id[q][strategy]["psnr"] is not None]
        result["metrics"][strategy] = {
            "mean_rank": float(values.mean()),
            "median_rank": float(np.median(values)),
            "mrr": float(np.mean(1.0 / values)),
            "recall_at_1": float(np.mean(values <= 1)),
            "recall_at_5": float(np.mean(values <= 5)),
            "recall_at_10": float(np.mean(values <= 10)),
            "mean_rank_shift_from_clean": float(shift.mean()),
            "median_rank_shift_from_clean": float(np.median(shift)),
            "asr_at_10_numerator": int(np.sum((values > 10) & eligible10)),
            "asr_at_10_denominator": int(np.sum(eligible10)),
            "mean_psnr": float(np.mean(psnr)),
            "mean_ssim": float(np.mean([
                paired_by_id[q][strategy]["ssim"] for q in query_ids
            ])),
            "mean_elapsed_seconds": float(np.mean([
                paired_by_id[q][strategy]["elapsed_time"] for q in query_ids
            ])),
        }
    raw_p_values = {}
    for baseline in strategies[:2]:
        delta = ranks[selected_strategy] - ranks[baseline]
        reciprocal_delta = (1.0 / ranks[selected_strategy] -
                            1.0 / ranks[baseline])
        wins = int(np.sum(delta > 0))
        ties = int(np.sum(delta == 0))
        losses = int(np.sum(delta < 0))
        p_value = exact_two_sided_sign_test(wins, losses)
        raw_p_values[baseline] = p_value
        result["paired_comparisons"][baseline] = {
            "direction": "positive rank delta means stronger target-rank suppression by minimum_unaffected_floor",
            "wins_ties_losses": [wins, ties, losses],
            "non_tied_denominator": wins + losses,
            "mean_rank_delta": float(delta.mean()),
            "median_rank_delta": float(np.median(delta)),
            "rank_delta_ci95_video_cluster": np.quantile([
                delta[index].mean() for index in bootstrap_samples
            ], [0.025, 0.975]).tolist(),
            "mean_mrr_delta": float(reciprocal_delta.mean()),
            "mrr_delta_ci95_video_cluster": np.quantile([
                reciprocal_delta[index].mean() for index in bootstrap_samples
            ], [0.025, 0.975]).tolist(),
            "exact_two_sided_sign_test_p_unadjusted": p_value,
        }
    for baseline, p_value in raw_p_values.items():
        adjusted = None if p_value is None else min(1.0, p_value * 2)
        result["paired_comparisons"][baseline].update({
            "bonferroni_family_size": 2,
            "bonferroni_adjusted_p": adjusted,
            "familywise_alpha": 0.05,
            "per_comparison_alpha": 0.025,
            "significant_familywise_0_05": adjusted is not None and adjusted <= 0.05,
        })
    return result


batch_summary = summarize(
    batch_verified["query_ids"], batch_verified["paired_by_id"],
    batch_verified["video_by_id"], 20260911)
hist_summary = summarize(
    hist_verified["query_ids"], hist_verified["paired_by_id"],
    hist_verified["video_by_id"], 20260908)

                                                                                     
for strategy in strategies:
    require(hist_summary["metrics"][strategy] == hist_audit["metrics"][strategy],
            f"Historical AG36 metric reproduction failed: {strategy}")
for baseline in strategies[:2]:
    require(hist_summary["paired_comparisons"][baseline] ==
            hist_audit["paired_comparisons"][baseline],
            f"Historical AG36 comparison reproduction failed: {baseline}")

combined_ids = (["historical::" + q for q in hist_verified["query_ids"]] +
                ["batch_a::" + q for q in batch_verified["query_ids"]])
combined_paired = {
    **{"historical::" + q: hist_verified["paired_by_id"][q]
       for q in hist_verified["query_ids"]},
    **{"batch_a::" + q: batch_verified["paired_by_id"][q]
       for q in batch_verified["query_ids"]},
}
combined_videos = {
    **{"historical::" + q: hist_verified["video_by_id"][q]
       for q in hist_verified["query_ids"]},
    **{"batch_a::" + q: batch_verified["video_by_id"][q]
       for q in batch_verified["query_ids"]},
}
require(len(combined_ids) == 141 and len(set(combined_ids)) == 141,
        "Combined cohort must contain 141 unique namespaced queries")
pooled_summary = summarize(combined_ids, combined_paired, combined_videos, 20260911)

                                                          
for baseline in strategies[:2]:
    compact = batch_comparison["summary"][baseline]
    audited = batch_summary["paired_comparisons"][baseline]
    require(compact["rank_better_equal_worse"] == audited["wins_ties_losses"] and
            compact["mean_rank_difference"] == audited["mean_rank_delta"],
            f"AG41 compact summary drift: {baseline}")

report = {
    "schema_version": 1,
    "status": "batch_a_and_pooled_post_evaluation_audit_passed",
    "artifact_class": "post_evaluation_cpu_integrity_statistics_and_paper_input",
    "source_hashes": {
        "batch_a_extension": EXPECTED[BATCH_SOURCE],
        "batch_a_extension_lock": EXPECTED[BATCH_LOCK],
        "batch_a_prompt_catalog": EXPECTED[BATCH_PROMPTS],
        "batch_a_prompt_manifest": EXPECTED[BATCH_PROMPT_MANIFEST],
        "batch_a_region_selection": EXPECTED[BATCH_SELECTION],
        "batch_a_pre_evaluation_lock": EXPECTED[BATCH_PRELOCK],
        "ag41_protocol": sha256(BATCH_PROTOCOL),
        "ag41_comparison": EXPECTED[BATCH_COMPARISON],
        "ag41_authorities": batch_verified["authority_hashes"],
        "historical_test48": EXPECTED[HIST_SOURCE],
        "historical_ag35_comparison": EXPECTED[HIST_COMPARISON],
        "historical_ag35_authorities": hist_verified["authority_hashes"],
        "historical_ag36_audit": EXPECTED[HIST_AUDIT],
    },
    "successful_execution_count": 423,
    "failure_count": 0,
    "strategy_count": 3,
    "inference_executed": False,
    "locked_inputs_modified": False,
    "uses_mock": False,
    "config_hashes": {
        "historical_test48": hist_verified["config_hash"],
        "batch_a_extension": batch_verified["config_hash"],
    },
    "batch_a_runtime_identities": batch_verified["runtime_identities"],
    "reporting_boundary": {
        "historical_test48": "previously observed organizer test48; reported separately",
        "batch_a_extension": "prospectively locked 93-query extension; primary new evaluation",
        "pooled_141": "secondary descriptive analysis only",
        "no_post_observation_tuning": True,
    },
    "cohorts": {
        "historical_test48": hist_summary,
        "batch_a_extension": batch_summary,
        "pooled_141_secondary": pooled_summary,
    },
}

payload = canonical_bytes(report)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
if OUTPUT.exists():
    require(OUTPUT.read_bytes() == payload, "Existing immutable AG42 audit differs")
else:
    temporary = OUTPUT.with_suffix(OUTPUT.suffix + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, OUTPUT)

print("AG42_BATCH_A_AND_POOLED_RESULT_AUDIT_OK")
print(json.dumps({
    "batch_a_extension": batch_summary,
    "historical_test48_reproduced": True,
    "pooled_141_secondary": pooled_summary,
    "successful_execution_count": 423,
    "failure_count": 0,
    "path": str(OUTPUT),
    "sha256": sha256(OUTPUT),
}, ensure_ascii=False, indent=2))


In [ ]:
if MODE in {"audit", "selected_stages"}:
    portable_root = Path("/content/reproduction_inputs")
    portable_root.mkdir(parents=True, exist_ok=True)
    for alias, source in [(portable_root / "artifacts", ARTIFACT_ROOT),
                          (portable_root / "dataset", DATA_ROOT)]:
        if not source.exists():
            raise FileNotFoundError(f"Configured input root does not exist: {source}")
        if alias.is_symlink() or alias.exists():
            if alias.resolve() != source:
                raise RuntimeError(f"Runtime alias already points elsewhere: {alias}")
        else:
            alias.symlink_to(source, target_is_directory=True)
    chosen = ["AG42"] if MODE == "audit" else SELECTED_STAGES
    if not chosen or len(chosen) != len(set(chosen)):
        raise ValueError("Select a nonempty list of distinct stages")
    if any(name not in STAGE_MANIFEST for name in chosen):
        raise ValueError("Unknown stage")
    harness_path = WORK_DIR / "execute_stage.py"
    harness_path.write_text(HARNESS, encoding="utf-8")
    for name in chosen:
        path = WORK_DIR / f"{name}.py"
        if hashlib.sha256(path.read_bytes()).hexdigest() != STAGE_MANIFEST[name]["notebook_source_sha256"]:
            raise ValueError(f"Stored stage changed: {name}")
        print("Executing", name, flush=True)
        subprocess.run([sys.executable, str(harness_path), str(path), str(runtime_path)], check=True)


## Interpreting the output

The supplied round-2 summary reports 61/32/0 wins/ties/losses versus target-centered and 19/74/0 versus highest-score selection. Mean paired rank gains are 12.65 and 1.66. ASR@10 is 32/58, 13/58 and 29/58 for minimum-floor, target-centered and highest-score selection respectively. These are descriptive success counts; the reported sign tests test rank signs, not binary ASR differences.

The video-cluster intervals are marginal 95% intervals; only the two sign-test p-values receive Bonferroni correction. Different target videos do not by themselves establish statistical independence. Whole-sequence PSNR/SSIM include unchanged frames and are not human perceptual judgments. Clean scores come from saved embeddings while attacked scores use fresh pixel encoding; the experiment has no zero-perturbation re-encoding control. Compression persistence and black-box transfer were not evaluated.

## Release checklist

Before public upload, provide authorized access instructions for the input artifacts and original authority files. Do not distribute private tokens or data without redistribution rights. Clear notebook outputs after any private run. The notebook is code-complete from the stated artifact boundary; it does not make unavailable data public or certify a new end-to-end GPU run.
